# Shell-and-Tube Heat Exchanger — Screening, Drawing-Geometry Import & Trusted-Reference Validation

| | |
|---|---|
| **Document type** | Engineering calculation notebook (preliminary screening) |
| **Release status** | 🔴 **BLOCKED — PRELIMINARY ONLY** (see §35 for live gate status) |
| **Units** | SI throughout (m, kg/s, °C/K, W/m²K, kPa); heat duty in kW |
| **Reproducibility** | Fixed random seed; environment recorded in §1.1 |
| **Software checks** | Automated verification suite in §33 |

> **Intended use.** Preliminary thermal-hydraulic screening and a gated workflow for importing real drawing geometry and comparing results against independent reference ratings. **Not** a validated design tool, **not** for fabrication or performance guarantees. Final design requires independent benchmark evidence and a qualified engineer's sign-off.

## Contents
| Part | Sections | Description |
|---|---|---|
| A | 1–9 | Synthetic dataset and ML regression comparison (learning illustration only) |
| B | 10–19 | LMTD, effectiveness, optimization, Re/Nu/fouling, multi-objective search |
| C | 20–24 | Design inputs, validation, constraint dashboard, sensitivity |
| D | 25–29 | Shell-side rating: Kern screening, Bell–Delaware, correlations, report export |
| E | 30–31 | Independent validation protocol, release gate, professional workflow pack |
| F | 32 | Drawing-geometry import and trusted-reference comparison |
| G | 33 | **Design-rule and applicability screening** against common heuristics |
| G+ | 33A | **Calculated rating closure**: computed F and U from film coefficients, resistance breakdown, area-margin robustness sweep |
| H | 34–36 | **Software verification suite**, **status dashboard** and **run manifest** (SHA-256 audit trail) |

## Revision history
| Rev | Change |
|---|---|
| A | Initial notebook |
| B | Fixed MLP target scaling (R² ≈ 0 → ≈ 0.999); turbulent-flow example inputs; stricter drawing-geometry validation; added cover, environment record, verification suite and status dashboard |
| C | Flow-regime label now consistent with the Re ≥ 10,000 Dittus–Boelter limit; added design-rule screening (§33), run manifest (§36); stored outputs cleared so every result comes from a fresh run |
| D | Added §33A: LMTD correction factor F computed (not assumed), U built from tube/shell films + fouling + wall, Gnielinski fallback for transitional tube flow, controlling-resistance table, area-margin robustness sweep, 14 new software tests, dashboard row |


## 1. Setup
Run this notebook from top to bottom. Install any missing packages in your notebook environment if needed.

In [ ]:
import warnings
import json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("xgboost is not installed; the notebook will use the sklearn MLP model and baseline.")

## 1.1 Environment and reproducibility record
Records software versions, platform and run time so results can be reproduced and audited.

In [ ]:
import sys, platform
from datetime import datetime, timezone
import sklearn, matplotlib

ENVIRONMENT_RECORD = {
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "xgboost": "available" if XGBOOST_AVAILABLE else "not installed (optional)",
    "random_state": RANDOM_STATE,
}
display(pd.DataFrame(ENVIRONMENT_RECORD.items(), columns=["Item", "Value"]))

## 2. Generate a reproducible simulated dataset
Assumptions used here: \(U\) is in W/(m²·K), area is in m², mass flow is in kg/s, and specific heat is in kJ/(kg·K). The calculated heat-transfer rate is reported in kW.

In [ ]:
N_SAMPLES = 1500
AREA_M2 = 12.0
CP_HOT = 4.184   # kJ/(kg K), illustrative water-like fluid
CP_COLD = 4.184 # kJ/(kg K), illustrative water-like fluid

rng = np.random.default_rng(RANDOM_STATE)

m_dot_hot = rng.uniform(1.0, 5.0, N_SAMPLES)
m_dot_cold = rng.uniform(1.2, 5.2, N_SAMPLES)
T_hot_in = rng.uniform(80.0, 100.0, N_SAMPLES)
T_cold_in = rng.uniform(20.0, 30.0, N_SAMPLES)

# Simulated overall heat-transfer coefficient, W/(m² K)
U = 550 + 45 * np.log(m_dot_hot * m_dot_cold) + rng.normal(0, 8, N_SAMPLES)

# Heat-transfer rate (kW): W converted to kW
Q = U * AREA_M2 * (T_hot_in - T_cold_in) / 1000.0

# Outlet temperatures from energy balance
T_hot_out = T_hot_in - Q / (m_dot_hot * CP_HOT)
T_cold_out = T_cold_in + Q / (m_dot_cold * CP_COLD)

# Illustrative pressure-drop correlations (kPa)
Delta_P_hot = 0.40 * m_dot_hot**1.85
Delta_P_cold = 0.42 * m_dot_cold**1.85

df = pd.DataFrame({
    "m_dot_hot_kg_s": m_dot_hot,
    "m_dot_cold_kg_s": m_dot_cold,
    "T_hot_in_C": T_hot_in,
    "T_cold_in_C": T_cold_in,
    "U_W_m2K": U,
    "T_hot_out_C": T_hot_out,
    "T_cold_out_C": T_cold_out,
    "Q_kW": Q,
    "Delta_P_hot_kPa": Delta_P_hot,
    "Delta_P_cold_kPa": Delta_P_cold,
})

assert df.notna().all().all(), "Dataset contains missing values."
assert (df["T_hot_in_C"] > df["T_cold_in_C"]).all(), "Hot inlet must be hotter than cold inlet."

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())
display(df.describe().T.round(3))

## 3. Engineering sanity checks
The two fluid-side heat rates should agree because both outlet temperatures are calculated from the same simulated heat duty.

In [ ]:
Q_hot_check = m_dot_hot * CP_HOT * (T_hot_in - T_hot_out)
Q_cold_check = m_dot_cold * CP_COLD * (T_cold_out - T_cold_in)

checks = pd.DataFrame({
    "check": ["Hot-side Q vs target", "Cold-side Q vs target"],
    "max_abs_error_kW": [
        np.max(np.abs(Q_hot_check - Q)),
        np.max(np.abs(Q_cold_check - Q)),
    ],
})
display(checks.round(8))

## 4. Train/test split and models
The overall heat-transfer coefficient \(U\) is included as a feature because it directly influences the simulated heat duty. The test set is held out from model fitting.

In [ ]:
FEATURES = [
    "m_dot_hot_kg_s", "m_dot_cold_kg_s",
    "T_hot_in_C", "T_cold_in_C", "U_W_m2K"
]
TARGET = "Q_kW"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    # The target (Q, ~hundreds of kW) must be scaled too, otherwise the MLP
    # fails to learn (R2 ~ 0). TransformedTargetRegressor handles this.
    "MLP neural network": TransformedTargetRegressor(
        regressor=Pipeline([
            ("scale", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="tanh",
                solver="adam",
                learning_rate_init=0.005,
                max_iter=1500,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ]),
        transformer=StandardScaler()
    ),
}
if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBRegressor(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=2
    )

predictions = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE (kW)": mean_absolute_error(y_test, pred),
        "RMSE (kW)": np.sqrt(mean_squared_error(y_test, pred)),
        "R²": r2_score(y_test, pred),
    })

metrics = pd.DataFrame(rows).sort_values("RMSE (kW)").reset_index(drop=True)
display(metrics.round(5))

## 5. Compare actual and predicted heat duty

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(6 * len(predictions), 5))
if len(predictions) == 1:
    axes = [axes]

low = min(float(y_test.min()), *(float(p.min()) for p in predictions.values()))
high = max(float(y_test.max()), *(float(p.max()) for p in predictions.values()))

for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.65, edgecolors="none")
    ax.plot([low, high], [low, high], "r--", linewidth=2, label="Ideal prediction")
    ax.set_title(name)
    ax.set_xlabel("Actual Q (kW)")
    ax.set_ylabel("Predicted Q (kW)")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Residual analysis
Residual = actual heat duty − predicted heat duty. A useful model should have small residuals without an obvious pattern.

In [ ]:
best_name = metrics.iloc[0]["Model"]
best_model = models[best_name]
best_pred = predictions[best_name]
residuals = y_test.to_numpy() - best_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(best_pred, residuals, alpha=0.65)
axes[0].axhline(0, linestyle="--", linewidth=2)
axes[0].set_title(f"Residuals — {best_name}")
axes[0].set_xlabel("Predicted Q (kW)")
axes[0].set_ylabel("Residual (kW)")
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals, bins=25, edgecolor="black")
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual (kW)")
axes[1].set_ylabel("Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best model by test RMSE: {best_name}")

## 7. Predict a new operating point
Edit the example values below. The values must use the same units as the dataset.

In [ ]:
def predict_heat_duty(m_dot_hot_kg_s, m_dot_cold_kg_s,
                      T_hot_in_C, T_cold_in_C, U_W_m2K,
                      model_name=None):
    """Predict simulated heat duty in kW for one operating point."""
    if m_dot_hot_kg_s <= 0 or m_dot_cold_kg_s <= 0:
        raise ValueError("Mass flow rates must be positive.")
    if T_hot_in_C <= T_cold_in_C:
        raise ValueError("Hot inlet temperature must exceed cold inlet temperature.")
    if U_W_m2K <= 0:
        raise ValueError("U must be positive.")

    selected_name = model_name or best_name
    if selected_name not in models:
        raise ValueError(f"Unknown model: {selected_name}")
    sample = pd.DataFrame([{
        "m_dot_hot_kg_s": m_dot_hot_kg_s,
        "m_dot_cold_kg_s": m_dot_cold_kg_s,
        "T_hot_in_C": T_hot_in_C,
        "T_cold_in_C": T_cold_in_C,
        "U_W_m2K": U_W_m2K,
    }], columns=FEATURES)

    predicted_q = float(models[selected_name].predict(sample)[0])
    theoretical_q = U_W_m2K * AREA_M2 * (T_hot_in_C - T_cold_in_C) / 1000.0
    return {
        "model": selected_name,
        "predicted_Q_kW": predicted_q,
        "simulation_equation_Q_kW": theoretical_q,
        "difference_kW": predicted_q - theoretical_q,
    }

example_prediction = predict_heat_duty(
    m_dot_hot_kg_s=2.5,
    m_dot_cold_kg_s=3.0,
    T_hot_in_C=90.0,
    T_cold_in_C=25.0,
    U_W_m2K=620.0,
)
display(pd.DataFrame([example_prediction]).round(4))

## 8. Export results
This saves the generated dataset, model comparison table, test predictions, and charts to the current notebook working directory.

In [ ]:
# Save tabular outputs
df.to_csv("heat_exchanger_simulated_data.csv", index=False)
metrics.to_csv("model_comparison_metrics.csv", index=False)

prediction_export = pd.DataFrame({
    "actual_Q_kW": y_test.to_numpy(),
    **{f"predicted_{name.replace(' ', '_')}_Q_kW": pred
       for name, pred in predictions.items()}
})
prediction_export.to_csv("test_set_predictions.csv", index=False)

# Save actual-vs-predicted comparison chart
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, best_pred, alpha=0.65)
ax.plot([low, high], [low, high], "r--", linewidth=2)
ax.set_title(f"Actual vs predicted heat duty — {best_name}")
ax.set_xlabel("Actual Q (kW)")
ax.set_ylabel("Predicted Q (kW)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("actual_vs_predicted.png", dpi=200, bbox_inches="tight")
plt.show()

print("Export complete:")
for filename in [
    "heat_exchanger_simulated_data.csv",
    "model_comparison_metrics.csv",
    "test_set_predictions.csv",
    "actual_vs_predicted.png",
]:
    print(" -", filename)

## 9. What this notebook does — and does not do
- Generates repeatable **synthetic** operating data.
- Checks the heat-duty energy balance used to create that data.
- Compares a mean baseline, an MLP neural network, and XGBoost when installed.
- Reports MAE, RMSE, and R² on a held-out test set.
- Provides a validated single-point prediction helper and exports CSV/PNG results.

**Important limitation:** this is not a true physics-informed neural network (PINN). The previous notebook's so-called physics loss simply penalized prediction error against labels; that is supervised learning, not an independent physics constraint. This upgraded version labels the neural network accurately and keeps the energy-balance checks explicit. For real engineering use, replace the simulated data/correlations with validated experimental or manufacturer data and include exchanger-specific constraints such as LMTD, effectiveness, fouling, and pressure-drop limits.

# 10. Interactive Heat Exchanger Calculator
Use the controls below in a Jupyter environment that supports `ipywidgets` (for example, JupyterLab with ipywidgets enabled). If widgets are unavailable, the engineering functions in the next sections can still be called directly from Python.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Use the calculation functions in the following sections directly.")

In [ ]:
def lmtd_counterflow(Th_in, Th_out, Tc_in, Tc_out):
    """Counter-current log mean temperature difference, K or °C difference."""
    dT1 = Th_in - Tc_out
    dT2 = Th_out - Tc_in
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError("Temperature cross/pinch detected: both terminal differences must be positive.")
    if np.isclose(dT1, dT2):
        return float(dT1)
    return float((dT1 - dT2) / np.log(dT1 / dT2))

def heat_duty_hot_kW(m_hot, cp_hot, Th_in, Th_out):
    return m_hot * cp_hot * (Th_in - Th_out)

def heat_duty_cold_kW(m_cold, cp_cold, Tc_in, Tc_out):
    return m_cold * cp_cold * (Tc_out - Tc_in)

def effectiveness_counterflow(m_hot, m_cold, cp_hot, cp_cold,
                              Th_in, Th_out, Tc_in, Tc_out):
    C_hot = m_hot * cp_hot
    C_cold = m_cold * cp_cold
    C_min = min(C_hot, C_cold)
    q_actual = min(
        heat_duty_hot_kW(m_hot, cp_hot, Th_in, Th_out),
        heat_duty_cold_kW(m_cold, cp_cold, Tc_in, Tc_out)
    )
    q_max = C_min * (Th_in - Tc_in)
    if q_max <= 0:
        raise ValueError("Hot inlet temperature must be above cold inlet temperature.")
    return float(q_actual / q_max)

def pressure_drop_power_law(m_dot, coefficient, exponent=1.85):
    """Illustrative pressure-drop estimate: ΔP = coefficient * mass_flow**exponent."""
    if m_dot <= 0 or coefficient < 0 or exponent <= 0:
        raise ValueError("Mass flow and exponent must be positive; coefficient cannot be negative.")
    return float(coefficient * m_dot**exponent)

def exchanger_performance(Th_in, Th_out, Tc_in, Tc_out,
                          m_hot, m_cold, cp_hot=4.184, cp_cold=4.184,
                          U=600.0, area=12.0,
                          dp_hot_coeff=0.40, dp_cold_coeff=0.42,
                          dp_exponent=1.85):
    """Calculate thermal performance and illustrative pressure drops. Cp in kJ/(kg K)."""
    if min(m_hot, m_cold, cp_hot, cp_cold, U, area) <= 0:
        raise ValueError("Mass flows, heat capacities, U, and area must be positive.")
    if Th_in <= Tc_in:
        raise ValueError("Hot inlet temperature must exceed cold inlet temperature.")
    q_hot = heat_duty_hot_kW(m_hot, cp_hot, Th_in, Th_out)
    q_cold = heat_duty_cold_kW(m_cold, cp_cold, Tc_in, Tc_out)
    lmtd = lmtd_counterflow(Th_in, Th_out, Tc_in, Tc_out)
    eps = effectiveness_counterflow(m_hot, m_cold, cp_hot, cp_cold,
                                    Th_in, Th_out, Tc_in, Tc_out)
    q_UA = U * area * lmtd / 1000.0
    return {
        "LMTD_K": lmtd,
        "Q_hot_side_kW": q_hot,
        "Q_cold_side_kW": q_cold,
        "Q_UA_estimate_kW": q_UA,
        "energy_balance_difference_kW": q_hot - q_cold,
        "effectiveness": eps,
        "effectiveness_percent": 100.0 * eps,
        "UA_kW_K": U * area / 1000.0,
        "DeltaP_hot_kPa_estimate": pressure_drop_power_law(m_hot, dp_hot_coeff, dp_exponent),
        "DeltaP_cold_kPa_estimate": pressure_drop_power_law(m_cold, dp_cold_coeff, dp_exponent),
    }

print("Thermal calculation functions loaded.")

## 11. Interactive inputs and live calculations
The default outlet temperatures are illustrative and should be adjusted to match the selected operating point. The calculator flags impossible terminal temperature differences.

In [ ]:
if WIDGETS_AVAILABLE:
    style = {"description_width": "initial"}
    layout = widgets.Layout(width="320px")

    Th_in_w = widgets.FloatSlider(value=90, min=40, max=150, step=1, description="Hot inlet (°C)", style=style, layout=layout)
    Th_out_w = widgets.FloatSlider(value=65, min=25, max=140, step=1, description="Hot outlet (°C)", style=style, layout=layout)
    Tc_in_w = widgets.FloatSlider(value=25, min=0, max=80, step=1, description="Cold inlet (°C)", style=style, layout=layout)
    Tc_out_w = widgets.FloatSlider(value=45, min=5, max=120, step=1, description="Cold outlet (°C)", style=style, layout=layout)
    mh_w = widgets.FloatSlider(value=2.5, min=0.1, max=10, step=0.1, description="Hot flow (kg/s)", style=style, layout=layout)
    mc_w = widgets.FloatSlider(value=3.0, min=0.1, max=10, step=0.1, description="Cold flow (kg/s)", style=style, layout=layout)
    cph_w = widgets.FloatSlider(value=4.184, min=0.5, max=6, step=0.01, description="Hot Cp (kJ/kg·K)", style=style, layout=layout)
    cpc_w = widgets.FloatSlider(value=4.184, min=0.5, max=6, step=0.01, description="Cold Cp (kJ/kg·K)", style=style, layout=layout)
    U_w = widgets.FloatSlider(value=600, min=50, max=2000, step=10, description="U (W/m²·K)", style=style, layout=layout)
    A_w = widgets.FloatSlider(value=12, min=0.5, max=100, step=0.5, description="Area (m²)", style=style, layout=layout)
    dph_w = widgets.FloatSlider(value=0.40, min=0.01, max=5, step=0.01, description="Hot ΔP coefficient", style=style, layout=layout)
    dpc_w = widgets.FloatSlider(value=0.42, min=0.01, max=5, step=0.01, description="Cold ΔP coefficient", style=style, layout=layout)
    exp_w = widgets.FloatSlider(value=1.85, min=1.0, max=3.0, step=0.05, description="ΔP exponent", style=style, layout=layout)

    calc_out = widgets.Output()

    def _run_live_calc(change=None):
        with calc_out:
            clear_output(wait=True)
            try:
                result = exchanger_performance(
                    Th_in_w.value, Th_out_w.value, Tc_in_w.value, Tc_out_w.value,
                    mh_w.value, mc_w.value, cph_w.value, cpc_w.value,
                    U_w.value, A_w.value, dph_w.value, dpc_w.value, exp_w.value
                )
                display(pd.DataFrame([result]).T.rename(columns={0: "Value"}).round(4))
                print("Reminder: ΔP values use illustrative power-law coefficients; replace with exchanger-specific validated correlations.")
            except ValueError as e:
                print("Input check:", e)

    controls = widgets.VBox([
        widgets.HTML("<h3>Inlet/outlet temperatures</h3>"),
        widgets.HBox([Th_in_w, Th_out_w]),
        widgets.HBox([Tc_in_w, Tc_out_w]),
        widgets.HTML("<h3>Flow, fluid properties and exchanger</h3>"),
        widgets.HBox([mh_w, mc_w]),
        widgets.HBox([cph_w, cpc_w]),
        widgets.HBox([U_w, A_w]),
        widgets.HTML("<h3>Illustrative pressure-drop correlation</h3>"),
        widgets.HBox([dph_w, dpc_w, exp_w]),
    ])
    for control in [Th_in_w, Th_out_w, Tc_in_w, Tc_out_w, mh_w, mc_w, cph_w, cpc_w, U_w, A_w, dph_w, dpc_w, exp_w]:
        control.observe(_run_live_calc, names="value")
    display(controls, calc_out)
    _run_live_calc()
else:
    print("Interactive controls need ipywidgets. Install it in your notebook environment, then restart the kernel.")

## 12. LMTD, effectiveness and pressure-drop analysis — notes
- LMTD uses the **counter-current** terminal temperature differences. For multipass or cross-flow equipment, use a correction factor \(F\): \(Q = U A F \Delta T_{lm}\).
- Effectiveness is calculated as \(\varepsilon = Q_{actual}/[C_{min}(T_{h,in}-T_{c,in})]\), using the smaller of the hot- and cold-side energy rates as the reported actual duty.
- Pressure drop currently uses a configurable **illustrative power law**, not a universal correlation. For real design, use validated geometry/fluid-specific methods (e.g., tube diameter/length, passes, viscosity, density, Reynolds number, fittings, fouling).
- Check energy-balance difference: a large mismatch between hot- and cold-side duties may indicate inconsistent measurements or unsuitable assumed values.

## 13. Optimization module (illustrative constrained search)
This grid-search optimizer chooses hot/cold mass flow rates to maximize the estimated heat duty while respecting user-supplied pressure-drop limits. It uses the notebook's illustrative correlations, so it is a demonstration, not a certified equipment sizing method.

In [ ]:
def optimize_flows(
    Th_in=90.0, Tc_in=25.0, U=600.0, area=12.0,
    cp_hot=4.184, cp_cold=4.184,
    hot_flow_bounds=(0.5, 5.0), cold_flow_bounds=(0.5, 5.0),
    max_dp_hot_kPa=8.0, max_dp_cold_kPa=8.0,
    dp_hot_coeff=0.40, dp_cold_coeff=0.42, dp_exponent=1.85,
    n_grid=60
):
    """Grid-search flow rates, maximizing a simple UA/energy-limited heat-duty estimate."""
    if Th_in <= Tc_in:
        raise ValueError("Hot inlet temperature must exceed cold inlet temperature.")
    if min(U, area, cp_hot, cp_cold, n_grid) <= 0:
        raise ValueError("U, area, heat capacities and n_grid must be positive.")
    h0, h1 = hot_flow_bounds
    c0, c1 = cold_flow_bounds
    if h0 <= 0 or c0 <= 0 or h1 <= h0 or c1 <= c0:
        raise ValueError("Flow bounds must be positive and upper bounds must exceed lower bounds.")

    hot_flows = np.linspace(h0, h1, int(n_grid))
    cold_flows = np.linspace(c0, c1, int(n_grid))
    records = []
    deltaT = Th_in - Tc_in
    UA_kW_K = U * area / 1000.0

    for mh in hot_flows:
        for mc in cold_flows:
            dp_h = pressure_drop_power_law(mh, dp_hot_coeff, dp_exponent)
            dp_c = pressure_drop_power_law(mc, dp_cold_coeff, dp_exponent)
            if dp_h > max_dp_hot_kPa or dp_c > max_dp_cold_kPa:
                continue
            C_hot = mh * cp_hot
            C_cold = mc * cp_cold
            C_min = min(C_hot, C_cold)
            C_max = max(C_hot, C_cold)
            Cr = C_min / C_max
            NTU = UA_kW_K / C_min
            # Counterflow effectiveness relation; handle Cr ~ 1 separately.
            if np.isclose(Cr, 1.0):
                eps = NTU / (1.0 + NTU)
            else:
                e = np.exp(-NTU * (1.0 - Cr))
                eps = (1.0 - e) / (1.0 - Cr * e)
            q = eps * C_min * deltaT
            records.append({
                "m_dot_hot_kg_s": mh,
                "m_dot_cold_kg_s": mc,
                "Q_est_kW": q,
                "effectiveness": eps,
                "DeltaP_hot_kPa_est": dp_h,
                "DeltaP_cold_kPa_est": dp_c,
                "NTU": NTU,
                "capacity_ratio_Cr": Cr,
            })

    if not records:
        raise ValueError("No feasible flow combinations. Increase pressure-drop limits or adjust bounds.")
    result = pd.DataFrame(records).sort_values("Q_est_kW", ascending=False).reset_index(drop=True)
    return result

optimization_results = optimize_flows()
print("Top 10 feasible flow combinations:")
display(optimization_results.head(10).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    optimization_results["m_dot_hot_kg_s"],
    optimization_results["m_dot_cold_kg_s"],
    c=optimization_results["Q_est_kW"],
    alpha=0.8
)
ax.set_title("Feasible operating points coloured by estimated heat duty")
ax.set_xlabel("Hot mass flow (kg/s)")
ax.set_ylabel("Cold mass flow (kg/s)")
ax.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax, label="Estimated Q (kW)")
plt.tight_layout()
plt.show()

## 14. Export optimization results

In [ ]:
optimization_results.to_csv("heat_exchanger_optimization_results.csv", index=False)
optimization_results.head(10).to_csv("heat_exchanger_optimization_top10.csv", index=False)
print("Saved heat_exchanger_optimization_results.csv and heat_exchanger_optimization_top10.csv")

# 15. Thermal-hydraulic calculations: Reynolds, Nusselt and fouling
This section adds educational, geometry-aware estimates for a shell-and-tube exchanger. Correlations are simplified screening models and must be replaced or verified against a selected TEMA configuration, fluid-property data, and an accepted design standard before real equipment sizing.

In [ ]:
def reynolds_number(rho_kg_m3, velocity_m_s, diameter_m, mu_pa_s):
    """Re = rho*v*D/mu; viscosity is dynamic viscosity in Pa·s."""
    if min(rho_kg_m3, velocity_m_s, diameter_m, mu_pa_s) <= 0:
        raise ValueError("Density, velocity, diameter and viscosity must be positive.")
    return rho_kg_m3 * velocity_m_s * diameter_m / mu_pa_s

def prandtl_number(cp_kj_kgK, mu_pa_s, k_w_mK):
    """Pr = cp*mu/k, with cp converted from kJ/(kg K) to J/(kg K)."""
    if min(cp_kj_kgK, mu_pa_s, k_w_mK) <= 0:
        raise ValueError("Cp, viscosity and thermal conductivity must be positive.")
    return (cp_kj_kgK * 1000.0) * mu_pa_s / k_w_mK

def nusselt_dittus_boelter(Re, Pr, heating_fluid=True):
    """Dittus-Boelter screening correlation for turbulent, fully developed tube flow.
    Usually applicable only when Re is sufficiently turbulent and properties are suitable.
    """
    if Re <= 0 or Pr <= 0:
        raise ValueError("Re and Pr must be positive.")
    n = 0.4 if heating_fluid else 0.3
    if Re < 10000:
        raise ValueError("Dittus-Boelter is intended for turbulent flow (typically Re >= 10,000).")
    return 0.023 * (Re ** 0.8) * (Pr ** n)

def tube_side_htc(Nu, k_w_mK, diameter_m):
    if min(Nu, k_w_mK, diameter_m) <= 0:
        raise ValueError("Nu, conductivity and diameter must be positive.")
    return Nu * k_w_mK / diameter_m

def overall_U_with_fouling(h_i, h_o, Rf_i=0.0002, Rf_o=0.0002,
                           wall_resistance_m2K_W=0.0001, diameter_ratio_do_di=1.0):
    """Approximate overall U on outer-area basis for a thin-wall screening model.
    Rf units are m² K/W. Diameter ratio allows a simple area-basis adjustment.
    """
    if min(h_i, h_o, diameter_ratio_do_di) <= 0:
        raise ValueError("Heat-transfer coefficients and diameter ratio must be positive.")
    if min(Rf_i, Rf_o, wall_resistance_m2K_W) < 0:
        raise ValueError("Fouling and wall resistances cannot be negative.")
    resistance = (
        1.0 / (diameter_ratio_do_di * h_i)
        + Rf_i / diameter_ratio_do_di
        + wall_resistance_m2K_W
        + Rf_o
        + 1.0 / h_o
    )
    return 1.0 / resistance

print("Reynolds, Prandtl, Nusselt, h and fouling-aware U functions loaded.")

## 16. Shell-and-tube / TEMA preliminary configuration
TEMA is a standards framework covering exchanger construction and mechanical details. The selector below records a preliminary front-head / shell / rear-head configuration; it does **not** certify TEMA compliance or replace mechanical design. Typical designation examples include BEM, AEL and AES, but suitability depends on maintenance, bundle removal, pressure, temperature, vibration and applicable code requirements.

In [ ]:
if WIDGETS_AVAILABLE:
    tema_front = widgets.Dropdown(
        options=[("A — Channel with removable cover", "A"),
                 ("B — Bonnet", "B"),
                 ("C — Channel integral with tubesheet", "C"),
                 ("N — Channel integral with tubesheet (special)", "N")],
        value="B", description="Front head"
    )
    tema_shell = widgets.Dropdown(
        options=[("E — One-pass shell", "E"),
                 ("F — Two-pass shell", "F"),
                 ("J — Divided-flow shell", "J"),
                 ("K — Kettle reboiler shell", "K"),
                 ("X — Cross-flow shell", "X")],
        value="E", description="Shell type"
    )
    tema_rear = widgets.Dropdown(
        options=[("M — Fixed tubesheet", "M"),
                 ("N — Fixed tubesheet variant", "N"),
                 ("P — Outside packed floating head", "P"),
                 ("S — Floating head with backing device", "S"),
                 ("T — Pull-through floating head", "T"),
                 ("U — U-tube bundle", "U")],
        value="M", description="Rear head"
    )
    tema_out = widgets.Output()
    def show_tema(change=None):
        with tema_out:
            clear_output(wait=True)
            code_name = tema_front.value + tema_shell.value + tema_rear.value
            display(pd.DataFrame([{
                "Preliminary TEMA-style designation": code_name,
                "Front head": tema_front.label if hasattr(tema_front, "label") else tema_front.value,
                "Shell": tema_shell.value,
                "Rear head": tema_rear.value,
                "Status": "Preliminary selection only — verify exact designation and construction rules"
            }]))
    for w in [tema_front, tema_shell, tema_rear]:
        w.observe(show_tema, names="value")
    display(widgets.VBox([tema_front, tema_shell, tema_rear]), tema_out)
    show_tema()
else:
    print("Install/enable ipywidgets to use the TEMA configuration selector.")

## 17. Geometry-based Reynolds / Nusselt / fouling example
The values below are illustrative. The tube-side velocity is calculated from total tube flow area, assuming the chosen number of tubes and tube passes.

In [ ]:
def tube_side_thermal_estimate(
    m_dot_kg_s, rho_kg_m3, mu_pa_s, cp_kj_kgK, k_w_mK,
    tube_inner_diameter_m, number_of_tubes, tube_passes=1,
    heating_fluid=True
):
    if min(m_dot_kg_s, rho_kg_m3, mu_pa_s, cp_kj_kgK, k_w_mK,
           tube_inner_diameter_m, number_of_tubes, tube_passes) <= 0:
        raise ValueError("All inputs must be positive.")
    area_one_tube = np.pi * tube_inner_diameter_m**2 / 4.0
    flow_area = area_one_tube * number_of_tubes / tube_passes
    velocity = m_dot_kg_s / (rho_kg_m3 * flow_area)
    Re = reynolds_number(rho_kg_m3, velocity, tube_inner_diameter_m, mu_pa_s)
    Pr = prandtl_number(cp_kj_kgK, mu_pa_s, k_w_mK)
    Nu = nusselt_dittus_boelter(Re, Pr, heating_fluid=heating_fluid)
    h_i = tube_side_htc(Nu, k_w_mK, tube_inner_diameter_m)
    return {
        "velocity_m_s": velocity,
        "Reynolds_number": Re,
        "Prandtl_number": Pr,
        "Nusselt_number": Nu,
        "tube_side_h_W_m2K": h_i
    }

try:
    thermal_example = tube_side_thermal_estimate(
        m_dot_kg_s=2.5, rho_kg_m3=997, mu_pa_s=0.00089,
        cp_kj_kgK=4.184, k_w_mK=0.60,
        tube_inner_diameter_m=0.016, number_of_tubes=30,
        tube_passes=2, heating_fluid=True
    )
    display(pd.DataFrame([thermal_example]).T.rename(columns={0: "Value"}).round(4))
except ValueError as err:
    print("Thermal estimate needs review:", err)

# Demonstrate fouling impact for illustrative film coefficients.
fouling_demo = pd.DataFrame([
    {"case": "Clean", "U_W_m2K": overall_U_with_fouling(2500, 1800, 0, 0, 0.0001, 1.1)},
    {"case": "Moderate fouling", "U_W_m2K": overall_U_with_fouling(2500, 1800, 0.0002, 0.0003, 0.0001, 1.1)},
    {"case": "Higher fouling", "U_W_m2K": overall_U_with_fouling(2500, 1800, 0.0005, 0.0007, 0.0001, 1.1)},
])
display(fouling_demo.round(2))

## 18. Multi-objective optimization: heat duty up, pressure drop and pumping power down
This grid-search approach reports a Pareto front instead of pretending there is one universally best design. Pumping power is estimated from volumetric flow × pressure drop / pump efficiency. The pressure-drop correlations remain illustrative. Real optimization should include allowable pressure limits, pump curves, mechanical limits, fouling, cost and a validated thermal design model.

In [ ]:
def multi_objective_optimize(
    Th_in=90.0, Tc_in=25.0, U=600.0, area=12.0,
    cp_hot=4.184, cp_cold=4.184,
    rho_hot=997.0, rho_cold=997.0,
    hot_flow_bounds=(0.5, 5.0), cold_flow_bounds=(0.5, 5.0),
    max_dp_hot_kPa=8.0, max_dp_cold_kPa=8.0,
    dp_hot_coeff=0.40, dp_cold_coeff=0.42, dp_exponent=1.85,
    pump_efficiency=0.70, n_grid=50
):
    if not 0 < pump_efficiency <= 1:
        raise ValueError("Pump efficiency must be in (0, 1].")
    if min(rho_hot, rho_cold) <= 0:
        raise ValueError("Densities must be positive.")
    candidates = optimize_flows(
        Th_in=Th_in, Tc_in=Tc_in, U=U, area=area,
        cp_hot=cp_hot, cp_cold=cp_cold,
        hot_flow_bounds=hot_flow_bounds, cold_flow_bounds=cold_flow_bounds,
        max_dp_hot_kPa=max_dp_hot_kPa, max_dp_cold_kPa=max_dp_cold_kPa,
        dp_hot_coeff=dp_hot_coeff, dp_cold_coeff=dp_cold_coeff,
        dp_exponent=dp_exponent, n_grid=n_grid
    ).copy()

    # kPa * m³/s = W; divide by efficiency for approximate electrical input.
    candidates["vol_flow_hot_m3_s"] = candidates["m_dot_hot_kg_s"] / rho_hot
    candidates["vol_flow_cold_m3_s"] = candidates["m_dot_cold_kg_s"] / rho_cold
    candidates["pump_power_hot_W"] = (
        candidates["DeltaP_hot_kPa_est"] * 1000 *
        candidates["vol_flow_hot_m3_s"] / pump_efficiency
    )
    candidates["pump_power_cold_W"] = (
        candidates["DeltaP_cold_kPa_est"] * 1000 *
        candidates["vol_flow_cold_m3_s"] / pump_efficiency
    )
    candidates["total_pump_power_W"] = (
        candidates["pump_power_hot_W"] + candidates["pump_power_cold_W"]
    )

    # Pareto: maximize Q; minimize both total pressure drop and pumping power.
    objectives = candidates[[
        "Q_est_kW", "DeltaP_hot_kPa_est",
        "DeltaP_cold_kPa_est", "total_pump_power_W"
    ]].to_numpy()
    is_pareto = np.ones(len(candidates), dtype=bool)
    for i, point in enumerate(objectives):
        dominates = (
            (objectives[:, 0] >= point[0]) &
            (objectives[:, 1] <= point[1]) &
            (objectives[:, 2] <= point[2]) &
            (objectives[:, 3] <= point[3]) &
            (
                (objectives[:, 0] > point[0]) |
                (objectives[:, 1] < point[1]) |
                (objectives[:, 2] < point[2]) |
                (objectives[:, 3] < point[3])
            )
        )
        if dominates.any():
            is_pareto[i] = False
    candidates["pareto_optimal"] = is_pareto
    return candidates.sort_values(
        ["pareto_optimal", "Q_est_kW"], ascending=[False, False]
    ).reset_index(drop=True)

multi_obj_results = multi_objective_optimize()
pareto_results = multi_obj_results[multi_obj_results["pareto_optimal"]].copy()

print(f"Feasible designs: {len(multi_obj_results)}")
print(f"Pareto-optimal designs: {len(pareto_results)}")
display(pareto_results.head(15).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    multi_obj_results["total_pump_power_W"],
    multi_obj_results["Q_est_kW"],
    alpha=0.25, label="Feasible designs"
)
ax.scatter(
    pareto_results["total_pump_power_W"],
    pareto_results["Q_est_kW"],
    alpha=0.9, label="Pareto front"
)
ax.set_xlabel("Estimated total pumping power (W)")
ax.set_ylabel("Estimated heat duty (kW)")
ax.set_title("Multi-objective trade-off: heat duty vs pumping power")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
multi_obj_results.to_csv("heat_exchanger_multi_objective_results.csv", index=False)
pareto_results.to_csv("heat_exchanger_pareto_front.csv", index=False)
print("Saved heat_exchanger_multi_objective_results.csv")
print("Saved heat_exchanger_pareto_front.csv")

## 19. Engineering limitations and safe use
- **TEMA selection:** the dropdown is a preliminary designation helper only. Verify designation letters and detailed requirements in the applicable TEMA edition and project specification.
- **Fouling:** fouling resistance depends on process fluid, temperature, chemistry, velocity, operating history and cleaning regime. Use project-approved values.
- **Re/Nu:** Dittus–Boelter is a screening correlation for suitable turbulent tube flow; it is not suitable for all laminar, transitional, developing, viscous or non-Newtonian conditions.
- **Pressure drop:** the power-law correlation is illustrative, not a universal shell-side or tube-side hydraulic design equation.
- **Optimization:** the Pareto front is only as credible as its heat-transfer and hydraulic models. Do not use these results as construction-ready sizing, code compliance, or safety certification.
- **Bell–Delaware-inspired factors:** the added J-factor framework is a transparent screening approximation, not the exact published method. Validate factors and hydraulic pressure drop with a qualified implementation/vendor.

# 20. Production-oriented shell-and-tube design workflow (engineering review required)
This section upgrades the notebook into a structured preliminary design and rating workflow with explicit inputs, thermal resistance accounting, tube-side pressure-drop screening, bundle geometry estimates, constraint flags, sensitivity/optimization support, and a report export. **It is not a code-certified design package**: production release requires review by a qualified heat-exchanger engineer and validation against the applicable TEMA edition, ASME BPVC Section VIII (where applicable), local regulations, and verified correlations/vendor data.

## Scope and model boundaries
Included here:
- User-editable design inputs and fluid properties.
- Tube-side flow area, velocity, Reynolds and Prandtl numbers, turbulent Nusselt estimate where applicable.
- Clean/fouled overall heat-transfer coefficient on a consistent approximate area basis.
- Counter-current LMTD and a configurable LMTD correction factor.
- Thermal duty and energy balance checks.
- Tube-side straight-tube friction pressure-drop screening plus a configurable minor-loss coefficient.
- Shell-side pressure-drop placeholder explicitly marked **not rated** unless a validated correlation is provided.
- Preliminary bundle/tube count, area, velocity and constraint report.
- A transparent candidate search with duty, pressure-drop and pumping-power objectives.
- CSV and JSON design report export.

Not included as certified calculations: full Bell–Delaware shell-side rating, baffle-window/crossflow leakage and bypass corrections, tube vibration, impingement protection, nozzle sizing, tube-sheet design, thermal expansion, U-tube/floating-head mechanical design, gasket/bolting, materials compatibility, fouling prediction, two-phase boiling/condensation, and code stress calculations.

## 21. Design input schema
Edit values in `DESIGN_INPUTS`. Keep SI units as labelled. Fluid properties should represent the actual bulk/film conditions required by your chosen method; this template uses constant properties as a preliminary approximation.

In [ ]:
DESIGN_INPUTS = {
    # Process conditions
    "hot_inlet_C": 90.0,
    "hot_outlet_target_C": 65.0,
    "cold_inlet_C": 25.0,
    "cold_outlet_target_C": 45.0,
    "hot_mass_flow_kg_s": 2.5,
    "cold_mass_flow_kg_s": 3.0,
    "hot_cp_kJ_kgK": 4.184,
    "cold_cp_kJ_kgK": 4.184,

    # Tube-side fluid properties (illustrative water-like values)
    "tube_side": "cold",  # "hot" or "cold"
    "tube_density_kg_m3": 997.0,
    "tube_viscosity_Pa_s": 0.00089,
    "tube_cp_kJ_kgK": 4.184,
    "tube_conductivity_W_mK": 0.60,

    # Preliminary geometry
    "tube_inner_diameter_m": 0.016,
    "tube_outer_diameter_m": 0.019,
    "tube_length_m": 4.0,
    "tube_count": 80,
    "tube_passes": 2,
    "shell_inside_diameter_m": 0.35,
    "baffle_spacing_m": 0.15,
    "baffle_cut_fraction": 0.25,
    "shell_passes": 1,

    # Shell-side screening properties / layout (illustrative water-like values)
    "shell_density_kg_m3": 997.0,
    "shell_viscosity_Pa_s": 0.00089,
    "shell_cp_kJ_kgK": 4.184,
    "shell_conductivity_W_mK": 0.60,
    "tube_pitch_m": 0.0254,  # centre-to-centre pitch; must exceed tube OD
    "tube_layout": "triangular",  # "triangular" or "square"
    "shell_roughness_m": 4.5e-5,

    # Bell-Delaware-inspired leakage / bypass geometry inputs (screening only)
    "shell_bundle_diametral_clearance_m": 0.006,  # shell ID minus bundle OD
    "tube_baffle_hole_diametral_clearance_m": 0.0008,
    "sealing_strip_pairs": 2,
    "baffle_spacing_inlet_m": 0.15,
    "baffle_spacing_outlet_m": 0.15,
    # Separate pressure-drop multipliers; set/calibrate from validated method/vendor data
    "shell_dp_leakage_multiplier": 0.85,
    "shell_dp_bypass_multiplier": 0.90,
    "shell_dp_end_spacing_multiplier": 1.00,

    # Heat transfer / resistance
    "U_clean_W_m2K": 900.0,
    "fouling_tube_m2K_W": 0.0002,
    "fouling_shell_m2K_W": 0.0003,
    "wall_resistance_m2K_W": 0.0001,
    "lmtd_correction_factor_F": 0.90,

    # Tube-side pressure drop inputs
    "tube_roughness_m": 1.5e-6,
    "tube_minor_loss_K_total": 4.0,
    "allowable_tube_dp_kPa": 50.0,
    "allowable_shell_dp_kPa": 30.0,
    "pump_efficiency": 0.70,

    # Preliminary acceptance targets
    "minimum_thermal_margin_fraction": 0.05,
}
print(json.dumps(DESIGN_INPUTS, indent=2))

## 22. Calculation functions and validation
The tube-side Darcy friction factor uses the laminar relation for Re < 2300 and the Swamee–Jain approximation for turbulent flow. The transitional range is flagged for engineering review. A separate Kern-style shell-side pressure-drop screening estimate is added later; it is not a validated Bell–Delaware rating and must not be treated as design-certified.

In [ ]:
def validate_design_inputs(p):
    required_positive = [
        "hot_mass_flow_kg_s", "cold_mass_flow_kg_s",
        "hot_cp_kJ_kgK", "cold_cp_kJ_kgK",
        "tube_density_kg_m3", "tube_viscosity_Pa_s",
        "tube_cp_kJ_kgK", "tube_conductivity_W_mK",
        "tube_inner_diameter_m", "tube_outer_diameter_m", "tube_length_m",
        "tube_count", "tube_passes", "shell_inside_diameter_m",
        "baffle_spacing_m", "U_clean_W_m2K", "lmtd_correction_factor_F",
        "pump_efficiency", "shell_density_kg_m3", "shell_viscosity_Pa_s",
        "shell_cp_kJ_kgK", "shell_conductivity_W_mK", "tube_pitch_m"
    ]
    for key in required_positive:
        if p[key] <= 0:
            raise ValueError(f"{key} must be positive.")
    if p["tube_pitch_m"] <= p["tube_outer_diameter_m"]:
        raise ValueError("Tube pitch must exceed tube outside diameter.")
    if p["tube_outer_diameter_m"] <= p["tube_inner_diameter_m"]:
        raise ValueError("Tube outer diameter must exceed inner diameter.")
    if p["tube_passes"] > p["tube_count"]:
        raise ValueError("Tube passes cannot exceed tube count in this preliminary model.")
    if not 0 < p["lmtd_correction_factor_F"] <= 1:
        raise ValueError("LMTD correction factor F must be in (0, 1].")
    if not 0 < p["baffle_cut_fraction"] < 0.5:
        raise ValueError("Baffle cut fraction must be between 0 and 0.5.")
    if not 0 < p["pump_efficiency"] <= 1:
        raise ValueError("Pump efficiency must be in (0, 1].")
    if p["hot_inlet_C"] <= p["cold_inlet_C"]:
        raise ValueError("Hot inlet must exceed cold inlet.")
    if p["hot_outlet_target_C"] >= p["hot_inlet_C"]:
        raise ValueError("Hot outlet target must be below hot inlet.")
    if p["cold_outlet_target_C"] <= p["cold_inlet_C"]:
        raise ValueError("Cold outlet target must exceed cold inlet.")
    return True

def darcy_friction_factor(Re, relative_roughness):
    if Re <= 0 or relative_roughness < 0:
        raise ValueError("Re must be positive and relative roughness non-negative.")
    if Re < 2300:
        return 64.0 / Re, "laminar"
    # Swamee–Jain approximation is for turbulent flow.
    f = 0.25 / (np.log10(relative_roughness / 3.7 + 5.74 / (Re ** 0.9)) ** 2)
    regime = "transitional-review" if Re < 10000 else "turbulent"  # consistent with Dittus-Boelter limit (Re >= 10,000)
    return float(f), regime

def lmtd_counter_current(Th_in, Th_out, Tc_in, Tc_out):
    dT_a = Th_in - Tc_out
    dT_b = Th_out - Tc_in
    if dT_a <= 0 or dT_b <= 0:
        raise ValueError(
            f"Temperature cross or non-positive terminal difference: ΔT1={dT_a:.3f} K, ΔT2={dT_b:.3f} K."
        )
    if np.isclose(dT_a, dT_b):
        return float(dT_a)
    return float((dT_a - dT_b) / np.log(dT_a / dT_b))

def calculate_shell_tube_preliminary(p):
    validate_design_inputs(p)
    # Target heat duties from each process side
    Q_hot_kW = p["hot_mass_flow_kg_s"] * p["hot_cp_kJ_kgK"] * (
        p["hot_inlet_C"] - p["hot_outlet_target_C"]
    )
    Q_cold_kW = p["cold_mass_flow_kg_s"] * p["cold_cp_kJ_kgK"] * (
        p["cold_outlet_target_C"] - p["cold_inlet_C"]
    )
    q_target_kW = min(Q_hot_kW, Q_cold_kW)

    lmtd = lmtd_counter_current(
        p["hot_inlet_C"], p["hot_outlet_target_C"],
        p["cold_inlet_C"], p["cold_outlet_target_C"]
    )
    F = p["lmtd_correction_factor_F"]
    deltaT_corrected = F * lmtd

    # Approximate fouled U from clean U and added fouling/wall resistance.
    # This is an overall-area-basis simplified resistance increment.
    R_clean = 1.0 / p["U_clean_W_m2K"]
    R_total = R_clean + p["fouling_tube_m2K_W"] + p["fouling_shell_m2K_W"] + p["wall_resistance_m2K_W"]
    U_fouled = 1.0 / R_total

    area_required_m2 = q_target_kW * 1000.0 / (U_fouled * deltaT_corrected)
    area_installed_m2 = np.pi * p["tube_outer_diameter_m"] * p["tube_length_m"] * p["tube_count"]
    area_margin_fraction = (area_installed_m2 - area_required_m2) / area_required_m2

    # Select tube-side flow based on chosen side.
    if p["tube_side"].lower() == "hot":
        m_tube = p["hot_mass_flow_kg_s"]
    elif p["tube_side"].lower() == "cold":
        m_tube = p["cold_mass_flow_kg_s"]
    else:
        raise ValueError("tube_side must be 'hot' or 'cold'.")

    di = p["tube_inner_diameter_m"]
    tubes_per_pass = p["tube_count"] / p["tube_passes"]
    total_flow_area = tubes_per_pass * np.pi * di**2 / 4.0
    velocity = m_tube / (p["tube_density_kg_m3"] * total_flow_area)
    Re = reynolds_number(p["tube_density_kg_m3"], velocity, di, p["tube_viscosity_Pa_s"])
    Pr = prandtl_number(p["tube_cp_kJ_kgK"], p["tube_viscosity_Pa_s"], p["tube_conductivity_W_mK"])

    if Re >= 10000:
        Nu = nusselt_dittus_boelter(Re, Pr, heating_fluid=True)
        Nu_status = "Dittus-Boelter screening estimate"
        h_tube = tube_side_htc(Nu, p["tube_conductivity_W_mK"], di)
    else:
        Nu = np.nan
        h_tube = np.nan
        Nu_status = "Not calculated: Dittus-Boelter not used below Re=10000"

    relative_roughness = p["tube_roughness_m"] / di
    f_darcy, flow_regime = darcy_friction_factor(Re, relative_roughness)
    # Total path length approximated as tube length * passes.
    path_length = p["tube_length_m"] * p["tube_passes"]
    dynamic_pressure_Pa = 0.5 * p["tube_density_kg_m3"] * velocity**2
    dp_friction_Pa = f_darcy * (path_length / di) * dynamic_pressure_Pa
    dp_minor_Pa = p["tube_minor_loss_K_total"] * dynamic_pressure_Pa
    dp_tube_kPa = (dp_friction_Pa + dp_minor_Pa) / 1000.0

    volumetric_flow_m3_s = m_tube / p["tube_density_kg_m3"]
    pumping_power_W = (dp_tube_kPa * 1000.0) * volumetric_flow_m3_s / p["pump_efficiency"]

    checks = {
        "terminal_temperature_check": "PASS",
        "tube_pressure_drop_check": "PASS" if dp_tube_kPa <= p["allowable_tube_dp_kPa"] else "FAIL",
        "shell_pressure_drop_check": "NOT RATED — validated shell-side method required",
        "area_margin_check": "PASS" if area_margin_fraction >= p["minimum_thermal_margin_fraction"] else "REVIEW",
        "flow_regime_check": "REVIEW" if flow_regime == "transitional-review" else flow_regime.upper(),
        "Nusselt_check": Nu_status,
        "mechanical_code_check": "NOT PERFORMED",
        "TEMA_compliance": "NOT CERTIFIED",
    }

    return {
        "Q_hot_target_kW": Q_hot_kW,
        "Q_cold_target_kW": Q_cold_kW,
        "Q_design_basis_kW": q_target_kW,
        "LMTD_counter_current_K": lmtd,
        "LMTD_correction_factor_F": F,
        "Corrected_LMTD_K": deltaT_corrected,
        "U_clean_W_m2K": p["U_clean_W_m2K"],
        "U_fouled_est_W_m2K": U_fouled,
        "Area_required_m2": area_required_m2,
        "Area_installed_m2": area_installed_m2,
        "Area_margin_percent": 100.0 * area_margin_fraction,
        "Tube_side_velocity_m_s": velocity,
        "Reynolds_number_tube_side": Re,
        "Prandtl_number_tube_side": Pr,
        "Nusselt_number_tube_side": Nu,
        "Tube_side_flow_regime": flow_regime,
        "Tube_side_friction_factor_Darcy": f_darcy,
        "Tube_side_pressure_drop_kPa_est": dp_tube_kPa,
        "Tube_side_pumping_power_W_est": pumping_power_W,
        "Shell_side_pressure_drop_kPa": np.nan,
        "Shell_side_status": "NOT RATED: validated shell-side correlation required",
        "Preliminary_bundle_area_m2": area_installed_m2,
        "checks": checks,
    }

preliminary_result = calculate_shell_tube_preliminary(DESIGN_INPUTS)
display(pd.DataFrame([
    {"Parameter": k, "Value": (json.dumps(v) if isinstance(v, dict) else v)}
    for k, v in preliminary_result.items()
]).round(4))

## 23. Constraint dashboard
A PASS here only means the simple screening check passed; it does not mean the exchanger is safe or code-compliant.

In [ ]:
constraint_rows = [
    {"Check": name, "Status": status}
    for name, status in preliminary_result["checks"].items()
]
constraint_df = pd.DataFrame(constraint_rows)
display(constraint_df)

## 24. Design sensitivity and preliminary multi-objective geometry search
This searches a small set of tube-count and tube-length candidates while holding the other inputs fixed. It reports feasible candidates with adequate estimated area margin and acceptable estimated tube-side pressure drop. This is a screening aid, not a substitute for full shell-side rating or mechanical design.

In [ ]:
def search_preliminary_geometry(
    base_inputs,
    tube_counts=(40, 60, 80, 100, 120, 160),
    tube_lengths_m=(2.0, 3.0, 4.0, 5.0, 6.0),
    max_tube_dp_kPa=None
):
    records = []
    for n_tubes in tube_counts:
        for length in tube_lengths_m:
            p = dict(base_inputs)
            p["tube_count"] = int(n_tubes)
            p["tube_length_m"] = float(length)
            try:
                r = calculate_shell_tube_preliminary(p)
            except (ValueError, ZeroDivisionError, FloatingPointError):
                continue
            dp_limit = max_tube_dp_kPa if max_tube_dp_kPa is not None else p["allowable_tube_dp_kPa"]
            area_ok = r["Area_margin_percent"] >= 100.0 * p["minimum_thermal_margin_fraction"]
            dp_ok = r["Tube_side_pressure_drop_kPa_est"] <= dp_limit
            records.append({
                "tube_count": n_tubes,
                "tube_length_m": length,
                "installed_area_m2": r["Area_installed_m2"],
                "required_area_m2": r["Area_required_m2"],
                "area_margin_percent": r["Area_margin_percent"],
                "tube_velocity_m_s": r["Tube_side_velocity_m_s"],
                "Re_tube": r["Reynolds_number_tube_side"],
                "tube_dp_kPa_est": r["Tube_side_pressure_drop_kPa_est"],
                "tube_pumping_power_W_est": r["Tube_side_pumping_power_W_est"],
                "area_constraint_pass": area_ok,
                "tube_dp_constraint_pass": dp_ok,
                "screening_feasible": area_ok and dp_ok,
            })
    result = pd.DataFrame(records)
    if result.empty:
        return result
    return result.sort_values(
        ["screening_feasible", "tube_pumping_power_W_est", "tube_dp_kPa_est"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

geometry_candidates = search_preliminary_geometry(DESIGN_INPUTS)
print("Top preliminary geometry candidates (screening only):")
display(geometry_candidates.head(15).round(3))

## 25. Shell-side preliminary hydraulic rating (Kern-style screening)
This section estimates shell-side crossflow area, equivalent diameter, mass velocity, Reynolds number and a rough Kern-style shell-side pressure drop. It is an initial hydraulic screen, not a validated Bell–Delaware rating. The next section adds a transparent **Bell–Delaware-inspired correction-factor framework** for baffle cut, leakage, bundle bypass, end spacing and low-Reynolds-number effects. These factors are approximate screening estimates and must be calibrated against a validated Bell–Delaware implementation or vendor software before design use.

In [ ]:
def calculate_shell_side_kern_screening(p):
    """Approximate shell-side hydraulic screen; not a Bell-Delaware rating."""
    Ds = float(p["shell_inside_diameter_m"])
    Do = float(p["tube_outer_diameter_m"])
    Pt = float(p.get("tube_pitch_m", 1.33 * Do))
    B = float(p["baffle_spacing_m"])
    rho = float(p.get("shell_density_kg_m3", 997.0))
    mu = float(p.get("shell_viscosity_Pa_s", 0.00089))
    rough = float(p.get("shell_roughness_m", 4.5e-5))
    layout = str(p.get("tube_layout", "triangular")).lower()
    if Ds <= 0 or Do <= 0 or B <= 0 or rho <= 0 or mu <= 0:
        raise ValueError("Shell diameter, tube OD, baffle spacing, density and viscosity must be positive.")
    if Pt <= Do:
        raise ValueError("Tube pitch must be greater than tube outside diameter.")
    if layout not in {"triangular", "square"}:
        raise ValueError("tube_layout must be 'triangular' or 'square'.")

    # Choose the fluid that is not routed through the tubes.
    if str(p["tube_side"]).lower() == "hot":
        m_shell = float(p["cold_mass_flow_kg_s"])
    elif str(p["tube_side"]).lower() == "cold":
        m_shell = float(p["hot_mass_flow_kg_s"])
    else:
        raise ValueError("tube_side must be 'hot' or 'cold'.")

    # Kern-style minimum crossflow area approximation at shell centreline.
    crossflow_area = Ds * B * (Pt - Do) / Pt
    if crossflow_area <= 0:
        raise ValueError("Calculated shell-side crossflow area is non-positive.")
    mass_velocity = m_shell / crossflow_area  # kg/(m² s)
    velocity = mass_velocity / rho
    if layout == "triangular":
        De = 1.10 * (Pt**2 - 0.917 * Do**2) / Do
    else:
        De = 4.0 * (Pt**2 - np.pi * Do**2 / 4.0) / (np.pi * Do)
    if De <= 0:
        raise ValueError("Equivalent diameter is non-positive; review tube pitch/layout.")
    Re_s = De * mass_velocity / mu

    # Screening friction factor: smooth turbulent/transition-style approximation.
    # Not suitable as a substitute for Bell-Delaware or vendor rating.
    if Re_s <= 0:
        raise ValueError("Shell-side Reynolds number must be positive.")
    if Re_s < 100:
        friction_factor = 64.0 / Re_s
        regime = "laminar — high uncertainty for shell crossflow"
    else:
        rel_rough = max(rough / De, 0.0)
        friction_factor = 0.25 / (np.log10(rel_rough / 3.7 + 5.74 / (Re_s**0.9))**2)
        regime = "transition/turbulent screening"

    # Number of idealized crossflow sections; baffle window and leakage losses omitted.
    n_cross = max(float(p["tube_length_m"]) / B - 1.0, 1.0)
    dp_pa = friction_factor * n_cross * (mass_velocity**2) / (2.0 * rho) * (Ds / De)
    dp_kpa = dp_pa / 1000.0
    pump_power_w = (dp_pa * (m_shell / rho)) / float(p["pump_efficiency"])
    return {
        "Shell_side_mass_flow_kg_s": m_shell,
        "Shell_crossflow_area_m2_est": crossflow_area,
        "Shell_equivalent_diameter_m_est": De,
        "Shell_mass_velocity_kg_m2s_est": mass_velocity,
        "Shell_velocity_m_s_est": velocity,
        "Shell_Reynolds_number_est": Re_s,
        "Shell_flow_regime_est": regime,
        "Shell_friction_factor_est": friction_factor,
        "Shell_crossflow_sections_est": n_cross,
        "Shell_side_pressure_drop_kPa_est": dp_kpa,
        "Shell_side_pumping_power_W_est": pump_power_w,
        "Shell_dp_allowable_kPa": float(p["allowable_shell_dp_kPa"]),
        "Shell_dp_screening_pass": bool(dp_kpa <= float(p["allowable_shell_dp_kPa"]))
    }

shell_side_result = calculate_shell_side_kern_screening(DESIGN_INPUTS)
print("Shell-side Kern-style screening (not a certified rating):")
display(pd.DataFrame([shell_side_result]).T.rename(columns={0: "Value"}))


## 26. Bell–Delaware-inspired correction factors and leakage/bypass screening

This module makes the main correction mechanisms explicit:
- **Jc**: baffle-cut / central-crossflow correction.
- **Jl**: shell-to-baffle and tube-to-baffle leakage correction, based on estimated leakage-area ratios.
- **Jb**: bundle bypass correction, influenced by estimated bypass-area ratio and sealing-strip pairs.
- **Js**: unequal inlet/outlet baffle-spacing correction.
- **Jr**: low-Reynolds-number correction flag/factor.

The estimates below are **Bell–Delaware-inspired screening approximations, not a full implementation of the published Bell–Delaware equations**. In particular, the area estimates simplify actual tube layout, baffle-window geometry, bundle perimeter, pass lanes, clearances and seal strips. Corrected heat-transfer coefficient is formed as `h_ideal × Jc × Jl × Jb × Js × Jr`; this is not a substitute for a full shell-side rating. Pressure-drop multipliers are separate user-calibrated inputs and are not inferred from heat-transfer factors. Do not use this module for final design without engineering validation.


In [ ]:
def calculate_bell_delaware_screening(p, kern_result=None):
    """Transparent Bell-Delaware-inspired screening factors; not a validated full rating."""
    if kern_result is None:
        kern_result = calculate_shell_side_kern_screening(p)

    Ds = float(p["shell_inside_diameter_m"])
    Do = float(p["tube_outer_diameter_m"])
    Pt = float(p["tube_pitch_m"])
    B = float(p["baffle_spacing_m"])
    cut = float(p["baffle_cut_fraction"])
    n_tubes = float(p["tube_count"])
    shell_bundle_clear = float(p.get("shell_bundle_diametral_clearance_m", 0.006))
    tube_hole_clear = float(p.get("tube_baffle_hole_diametral_clearance_m", 0.0008))
    seal_pairs = int(p.get("sealing_strip_pairs", 0))
    Bin = float(p.get("baffle_spacing_inlet_m", B))
    Bout = float(p.get("baffle_spacing_outlet_m", B))
    if min(Ds, Do, Pt, B, n_tubes) <= 0:
        raise ValueError("Shell diameter, tube OD, pitch, spacing and tube count must be positive.")
    if shell_bundle_clear < 0 or tube_hole_clear < 0 or seal_pairs < 0:
        raise ValueError("Clearances and sealing-strip pairs cannot be negative.")
    if not 0 < cut < 0.5:
        raise ValueError("Baffle cut fraction must be between 0 and 0.5.")
    if min(Bin, Bout) <= 0:
        raise ValueError("Inlet and outlet baffle spacing must be positive.")

    # Simplified geometry proxies (not exact Bell–Delaware area equations).
    bundle_diam = max(Ds - shell_bundle_clear, 0.0)
    shell_bundle_gap_area = np.pi * Ds * shell_bundle_clear / 2.0
    tube_baffle_leak_area = n_tubes * np.pi * Do * tube_hole_clear / 2.0
    central_crossflow_area = float(kern_result["Shell_crossflow_area_m2_est"])
    leakage_area = shell_bundle_gap_area + tube_baffle_leak_area
    leakage_ratio = float(np.clip(leakage_area / max(leakage_area + central_crossflow_area, 1e-12), 0.0, 0.95))

    # Bypass-area proxy: shell-to-bundle gap divided by a simplified total flow-area proxy.
    bypass_area_proxy = np.pi * Ds * shell_bundle_clear / 2.0
    bypass_ratio = float(np.clip(bypass_area_proxy / max(bypass_area_proxy + central_crossflow_area, 1e-12), 0.0, 0.95))

    # Transparent empirical screening factors. These are not the exact published correlations.
    Jc = float(np.clip(0.55 + 0.72 * (1.0 - 2.0 * cut), 0.50, 1.0))
    Jl = float(np.clip(1.0 - 0.8 * leakage_ratio, 0.50, 1.0))
    sealing_effectiveness = 1.0 - np.exp(-0.7 * seal_pairs)
    Jb = float(np.clip(1.0 - bypass_ratio * (1.0 - sealing_effectiveness), 0.50, 1.0))
    mean_spacing = (Bin + Bout) / 2.0
    spacing_ratio = min(Bin, Bout) / max(mean_spacing, 1e-12)
    Js = float(np.clip(0.6 + 0.4 * spacing_ratio, 0.50, 1.0))
    Re_s = float(kern_result["Shell_Reynolds_number_est"])
    Jr = 1.0 if Re_s >= 100.0 else float(np.clip(Re_s / 100.0, 0.20, 1.0))

    factors = {"Jc_baffle_cut": Jc, "Jl_leakage": Jl, "Jb_bundle_bypass": Jb,
               "Js_end_spacing": Js, "Jr_low_Re": Jr}
    J_total = float(np.prod(list(factors.values())))

    # User-calibrated hydraulic multipliers are kept separate from heat-transfer factors.
    dp_leak = float(p.get("shell_dp_leakage_multiplier", 1.0))
    dp_bypass = float(p.get("shell_dp_bypass_multiplier", 1.0))
    dp_ends = float(p.get("shell_dp_end_spacing_multiplier", 1.0))
    if min(dp_leak, dp_bypass, dp_ends) <= 0:
        raise ValueError("Shell-side pressure-drop multipliers must be positive.")
    dp_corrected = float(kern_result["Shell_side_pressure_drop_kPa_est"] * dp_leak * dp_bypass * dp_ends)

    # Use supplied clean U only as a proxy for an idealized shell-side h; do not claim overall U.
    h_ideal_proxy = float(p.get("U_clean_W_m2K", np.nan))
    h_corrected_proxy = h_ideal_proxy * J_total if np.isfinite(h_ideal_proxy) else np.nan

    return {
        "Shell_bundle_diameter_proxy_m": bundle_diam,
        "Shell_bundle_gap_area_proxy_m2": shell_bundle_gap_area,
        "Tube_baffle_leakage_area_proxy_m2": tube_baffle_leak_area,
        "Estimated_leakage_area_ratio": leakage_ratio,
        "Estimated_bundle_bypass_area_ratio": bypass_ratio,
        **factors,
        "J_total_screening_product": J_total,
        "Ideal_h_proxy_W_m2K_NOT_shell_h": h_ideal_proxy,
        "Corrected_h_proxy_W_m2K_NOT_shell_h": h_corrected_proxy,
        "Shell_dp_leakage_multiplier_user": dp_leak,
        "Shell_dp_bypass_multiplier_user": dp_bypass,
        "Shell_dp_end_spacing_multiplier_user": dp_ends,
        "Shell_side_pressure_drop_kPa_adjusted_screening": dp_corrected,
        "Pressure_drop_adjusted_pass": bool(dp_corrected <= float(p["allowable_shell_dp_kPa"])),
        "Model_status": "SCREENING ONLY — NOT FULL BELL–DELAWARE"
    }

bell_delaware_screening = calculate_bell_delaware_screening(DESIGN_INPUTS, shell_side_result)
print("Bell–Delaware-inspired correction-factor screen (approximate, not a full rating):")
display(pd.DataFrame([bell_delaware_screening]).T.rename(columns={0: "Value"}))


## 27. Correlation-based shell-side thermal/hydraulic rating (engineering review required)

This section replaces the earlier arbitrary `U_clean`-as-shell-film proxy with a **separate shell-side film coefficient**. It uses a documented tube-bank Nusselt correlation for the ideal crossflow coefficient, applies explicit Bell–Delaware correction-factor equations, and builds a transparent crossflow/window/end-zone pressure-drop estimate.

**Scope and validation status:** this is a literature-correlation implementation and software-smoke-tested, but it is **not independently validated against vendor software, experimental data, or a benchmark design**. Exact Bell–Delaware rating requires detailed baffle-window geometry, tube-row counts, bundle layout, clearances, wall viscosity, and the applicable friction-factor tables/correlations. The pressure-drop model here remains a screening implementation where ideal bundle friction and window geometry are approximated. Do not use for final equipment purchase, guarantees, safety cases, or code compliance without qualified review and benchmark comparison.

Correlation basis: Bell–Delaware correction structure and tube-bank correlation documented in the NeqSim API; correction-factor equations and pressure-drop component structure are also described in the cited literature linked in the notebook notes below.


In [ ]:
# Correlation-based shell-side coefficient and pressure-drop screening.
# SI units throughout. This is NOT a certified full Bell–Delaware implementation.

def _clip_factor(value, lo=0.0, hi=1.2):
    return float(np.clip(value, lo, hi))

def calculate_shell_side_correlation_rating(p, base_screen=None, bd_screen=None):
    """Calculate shell-side film h and a componentized pressure-drop estimate.

    Ideal h: Zhukauskas-style tube-bank correlation as documented by NeqSim:
      staggered/triangular: Nu = 0.35*(Pt/Do)^0.2*Re^0.6*Pr^0.36*(Pr/Pr_w)^0.25
      inline/square:        Nu = 0.27*Re^0.63*Pr^0.36*(Pr/Pr_w)^0.25
    The correlation's exact applicability depends on geometry and Re; review source/reference.

    Correction factors use standard Bell–Delaware structure. Some geometry terms are
    estimated from the limited inputs in this notebook, so results are marked screening.
    """
    if base_screen is None:
        base_screen = calculate_shell_side_kern_screening(p)
    if bd_screen is None:
        bd_screen = calculate_bell_delaware_screening(p, base_screen)

    Do = float(p['tube_outer_diameter_m'])
    Pt = float(p['tube_pitch_m'])
    Ds = float(p['shell_inside_diameter_m'])
    B = float(p['baffle_spacing_m'])
    rho = float(p['shell_density_kg_m3'])
    mu = float(p['shell_viscosity_Pa_s'])
    cp = float(p['shell_cp_kJ_kgK']) * 1000.0
    k = float(p['shell_conductivity_W_mK'])
    m_shell = float(p['hot_mass_flow_kg_s'] if str(p['tube_side']).lower() == 'cold' else p['cold_mass_flow_kg_s'])
    mu_wall = float(p.get('shell_wall_viscosity_Pa_s', mu))
    if min(Do, Pt, Ds, B, rho, mu, cp, k, mu_wall, m_shell) <= 0:
        raise ValueError('Shell-side geometry, flow and fluid properties must be positive.')
    if Pt <= Do:
        raise ValueError('Tube pitch must exceed tube outside diameter.')

    Sm = float(base_screen['Shell_crossflow_area_m2_est'])
    Gs = m_shell / Sm
    Re_o = Do * Gs / mu
    Pr = cp * mu / k
    Pr_w = cp * mu_wall / k
    layout = str(p.get('tube_layout', 'triangular')).lower()
    if layout not in ('triangular', 'square'):
        raise ValueError("tube_layout must be 'triangular' or 'square'.")

    if layout == 'triangular':
        Nu_ideal = 0.35 * (Pt / Do)**0.2 * Re_o**0.6 * Pr**0.36 * (Pr / Pr_w)**0.25
        ideal_corr = 'Zhukauskas-style staggered tube-bank correlation (triangular layout)'
    else:
        Nu_ideal = 0.27 * Re_o**0.63 * Pr**0.36 * (Pr / Pr_w)**0.25
        ideal_corr = 'Zhukauskas-style inline tube-bank correlation (square layout)'
    h_ideal = Nu_ideal * k / Do

    # Geometry-derived factors. Jc uses estimated crossflow fraction; Jl uses leakage-area
    # ratios; Jb uses bypass fraction and sealing strips; Js uses end/central spacings;
    # Jr applies the common low-Re adverse-gradient form.
    Fc = float(np.clip(1.0 - 2.0 * float(p['baffle_cut_fraction']), 0.05, 0.95))
    Jc = 0.55 + 0.72 * Fc
    cross_area = Sm
    shell_gap_area = float(bd_screen['Shell_bundle_gap_area_proxy_m2'])
    tube_leak_area = float(bd_screen['Tube_baffle_leakage_area_proxy_m2'])
    leak_area = shell_gap_area + tube_leak_area
    r_l = float(np.clip(leak_area / max(cross_area, 1e-12), 0.0, 5.0))
    r_s = float(np.clip(shell_gap_area / max(leak_area, 1e-12), 0.0, 1.0))
    Jl = 0.44 * (1.0 - r_s) + (1.0 - 0.44 * (1.0 - r_s)) * np.exp(-2.2 * r_l)

    bypass_area = float(bd_screen['Shell_bundle_gap_area_proxy_m2'])
    Fsbp = float(np.clip(bypass_area / max(bypass_area + cross_area, 1e-12), 0.0, 0.95))
    n_rows = max(int((Ds * (1.0 - float(p['baffle_cut_fraction'])) / max(Pt, 1e-12))), 1)
    n_cl = max(n_rows, 1)
    seal_pairs = max(int(p.get('sealing_strip_pairs', 0)), 0)
    ratio_ss = min(2.0 * seal_pairs / n_cl, 0.5)
    Cbh = 1.35 if Re_o <= 100.0 else 1.25
    Jb = float(np.exp(-Cbh * Fsbp * (1.0 - ratio_ss**(1.0/3.0)))) if ratio_ss < 0.5 else 1.0

    Bin = float(p.get('baffle_spacing_inlet_m', B))
    Bout = float(p.get('baffle_spacing_outlet_m', B))
    n_baffles = max(int(np.floor(float(p['tube_length_m']) / B) - 1), 1)
    n_exp = 0.2 if Re_o > 100.0 else 1.0
    Js_num = (n_baffles - 1.0) + (Bin / B)**(1.0 - n_exp) + (Bout / B)**(1.0 - n_exp)
    Js_den = (n_baffles - 1.0) + Bin / B + Bout / B
    Js = float(np.clip(Js_num / max(Js_den, 1e-12), 0.5, 1.2))
    if Re_o > 100.0:
        Jr = 1.0
    else:
        Jr1 = (10.0 / max(n_rows, 1))**0.18
        Jr = float(np.clip(Jr1 if Re_o <= 20.0 else Jr1 + ((20.0 - Re_o) / 80.0) * (Jr1 - 1.0), 0.2, 2.0))

    J_factors = {'Jc_baffle_cut': float(Jc), 'Jl_leakage': float(Jl), 'Jb_bundle_bypass': float(Jb),
                 'Js_end_spacing': float(Js), 'Jr_low_Re': float(Jr)}
    h_shell = float(h_ideal * np.prod(list(J_factors.values())))

    # Pressure drop component estimate: ideal crossflow pressure loss from the existing
    # Kern-style screen is split into central sections; Bell–Delaware-style Rl/Rb factors
    # are applied. Window and end-zone losses are estimated separately from area/velocity.
    # These approximations are intentionally visible and must be benchmarked before use.
    dp_ideal_total_Pa = float(base_screen['Shell_side_pressure_drop_kPa_est']) * 1000.0
    Rl = float(np.clip(0.4 + 0.6 * np.exp(-2.0 * r_l), 0.2, 1.0))
    Rb = float(np.clip(np.exp(-Cbh * Fsbp * (1.0 - ratio_ss**(1.0/3.0))), 0.2, 1.0))
    dp_cross_Pa = dp_ideal_total_Pa * Rl * Rb

    # Segmental-window area proxy from the circular shell segment; excludes tube blockage,
    # so a conservative open-area adjustment is applied and documented as approximate.
    cut_height = float(p['baffle_cut_fraction']) * Ds
    y = np.clip(Ds / 2.0 - cut_height, -Ds/2.0, Ds/2.0)
    theta = np.arccos(np.clip(y / (Ds/2.0), -1.0, 1.0))
    segment_area = (Ds**2 / 4.0) * (theta - np.sin(theta) * np.cos(theta))
    window_area = max(2.0 * segment_area * 0.65, 1e-6)  # approximate net free area
    window_mass_velocity = m_shell / window_area
    dp_window_per_zone_Pa = 0.5 * (window_mass_velocity**2) / rho * 1.5
    dp_window_Pa = n_baffles * dp_window_per_zone_Pa * Rl
    end_area = max(Sm, 1e-6)
    end_velocity = m_shell / (rho * end_area)
    dp_end_Pa = 2.0 * (0.5 * rho * end_velocity**2) * (Bin / B + Bout / B) * Rb
    dp_shell_Pa = dp_cross_Pa + dp_window_Pa + dp_end_Pa

    checks = {
        'positive_geometry_and_properties': 'PASS',
        'tube_bank_Re_range': 'REVIEW — verify correlation applicability' if not (10 <= Re_o <= 1e6) else 'IN CORRELATION RANGE (verify layout-specific limits)',
        'shell_dp_limit': 'PASS' if dp_shell_Pa / 1000.0 <= float(p['allowable_shell_dp_kPa']) else 'FAIL',
        'model_validation': 'NOT INDEPENDENTLY VALIDATED — benchmark required',
    }
    return {
        'Model_status': 'CORRELATION-BASED SCREENING — NOT CERTIFIED',
        'Ideal_heat_transfer_correlation': ideal_corr,
        'Shell_crossflow_area_m2': Sm,
        'Shell_mass_velocity_kg_m2s': Gs,
        'Shell_Reynolds_number_based_on_tube_OD': Re_o,
        'Shell_Prandtl_number': Pr,
        'Ideal_tube_bank_Nusselt_number': float(Nu_ideal),
        'Ideal_shell_film_coefficient_W_m2K': float(h_ideal),
        **J_factors,
        'Bell_Delaware_correction_product': float(np.prod(list(J_factors.values()))),
        'Corrected_shell_side_film_coefficient_W_m2K': h_shell,
        'Pressure_drop_leakage_factor_Rl_est': Rl,
        'Pressure_drop_bypass_factor_Rb_est': Rb,
        'Pressure_drop_central_crossflow_kPa_est': dp_cross_Pa / 1000.0,
        'Pressure_drop_window_zones_kPa_est': dp_window_Pa / 1000.0,
        'Pressure_drop_inlet_outlet_zones_kPa_est': dp_end_Pa / 1000.0,
        'Shell_side_total_pressure_drop_kPa_est': dp_shell_Pa / 1000.0,
        'Shell_side_pumping_power_W_est': dp_shell_Pa * (m_shell / rho) / float(p['pump_efficiency']),
        'Shell_dp_allowable_kPa': float(p['allowable_shell_dp_kPa']),
        'checks': checks,
        'limitations': [
            'Not independently validated against vendor software, experimental data or a benchmark design.',
            'Crossflow ideal pressure drop inherits the simplified Kern screening model.',
            'Window area ignores exact tube blockage and window tube-row count.',
            'Leakage and bypass areas are estimated from simplified clearance geometry.',
            'Use a validated Bell–Delaware implementation and qualified engineering review for final design.'
        ]
    }

shell_side_correlation_rating = calculate_shell_side_correlation_rating(
    DESIGN_INPUTS, shell_side_result, bell_delaware_screening
)
print('Correlation-based shell-side thermal/hydraulic rating (screening):')
display(pd.DataFrame([{
    k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)
    for k, v in shell_side_correlation_rating.items()
}]).T.rename(columns={0: 'Value'}))


### Correlation references and validation checklist

- NeqSim API documentation, `BellDelawareMethod`: documents the ideal tube-bank correlation option, correction-factor structure, and shell-side pressure-drop components: https://equinor.github.io/neqsimhome/javadoc/site/apidocs/neqsim/process/mechanicaldesign/heatexchanger/BellDelawareMethod.html
- Bell, K. J. (1981), “Delaware Method for Shell-Side Design,” in *Heat Exchangers: Thermal-Hydraulic Fundamentals and Design*.
- Gonçalves et al., “Linear Method for the Design and Optimization of Shell-and-Tube Heat Exchangers Using the Bell–Delaware Method,” AIChE Journal (correction-factor and pressure-drop equation overview): https://www.ou.edu/class/che-design/pub-papers/Linear%20method%20for%20the%20design%20and%20optimization%20of%20shell%20and%20tube%20heat%20exchangers%20using%20the%20Bell-Delaware%20Method%28Goncalves%20et%20al%29-19.pdf

**Before calling a result validated:** compare against an independent commercial or open-source implementation for the same full geometry and properties; document error tolerances; test limiting cases; and have a qualified heat-exchanger engineer sign off. This notebook currently performs software smoke tests only.


## 28. Geometry-aware Bell–Delaware equation set and benchmark harness

This section advances the previous screen by deriving key geometry quantities from the documented DWSIM Advanced Heat Exchanger equations: bundle outer diameter from shell-to-baffle clearance, centreline crossflow area, shell-to-baffle leakage area, tube-to-baffle leakage area, baffle count, and an estimated net window area. It then uses the documented component structure:

- \(\Delta P_{id}=2 f_i N_{cl} G_s^2/\rho_s\)
- \(\Delta P_{cross}=(N_b-1)\Delta P_{id}R_lR_b\)
- \(\Delta P_{window}=N_b(2+0.6N_{tw})G_w^2/(2\rho_s)R_l\), with \(G_w=\dot m_s/\sqrt{S_m A_w}\)
- \(\Delta P_{end}=2\Delta P_{id}(1+N_{cw}/N_{cl})R_b\)

**Important:** the ideal tube-bank friction factor \(f_i\) and row counts are normally obtained from layout/Reynolds-specific published tables and detailed geometry. Because those tables and exact CAD geometry are not supplied in this notebook, the defaults below are explicitly *screening estimates* and can be overridden with values from a trusted implementation/vendor. The window area is also an estimate (gross circular segment less an estimated tube blockage). This section does **not** make the model validated by itself.

Source equations: [DWSIM Advanced Heat Exchanger documentation](https://dwsim.org/help/main/24-unit-operations.html) and [NeqSim BellDelawareMethod API](https://equinor.github.io/neqsimhome/javadoc/site/apidocs/neqsim/process/mechanicaldesign/heatexchanger/BellDelawareMethod.html).

In [ ]:
# Optional advanced overrides. Replace estimates with values from the actual exchanger drawing
# or a trusted Bell–Delaware implementation whenever available.
BELL_DELAWARE_ADVANCED = {
    "bundle_outer_diameter_m": None,        # if known from drawing; else Ds - shell/baffle diametral clearance
    "tube_rows_crossflow_override": None,   # N_cl from actual layout / trusted software
    "tube_rows_window_override": None,      # N_tw from actual layout / trusted software
    "tube_rows_endzone_override": None,     # N_cw from actual layout / trusted software
    "ideal_bank_friction_factor_fi": 0.03,  # screening placeholder only; replace with tabulated/correlation value
    "reference_h_shell_W_m2K": None,        # optional reference result from DWSIM/vendor
    "reference_dp_shell_kPa": None,         # optional reference result from DWSIM/vendor
    "reference_source_label": "Not supplied"
}
print(json.dumps(BELL_DELAWARE_ADVANCED, indent=2))

In [ ]:
def _circle_segment_area_from_height(D, h):
    """Area of a circular segment of height h in a circle of diameter D (m²)."""
    R = D / 2.0
    h = float(np.clip(h, 0.0, D))
    if h <= 0:
        return 0.0
    if h >= D:
        return float(np.pi * R**2)
    return float(R**2 * np.arccos((R-h)/R) - (R-h)*np.sqrt(max(2*R*h-h*h, 0.0)))


def calculate_geometry_aware_bell_delaware(p, adv=None):
    """Geometry-informed single-phase shell-side rating with explicit estimate flags.

    Equations follow the published Bell–Delaware component structure. Where exact row
    counts, ideal-bank friction-factor tables, or CAD-derived window free area are missing,
    transparent estimates are used and surfaced in the result status.
    """
    adv = dict(BELL_DELAWARE_ADVANCED if adv is None else adv)
    Ds = float(p["shell_inside_diameter_m"])
    Do = float(p["tube_outer_diameter_m"])
    Pt = float(p["tube_pitch_m"])
    B = float(p["baffle_spacing_m"])
    cut = float(p["baffle_cut_fraction"])
    Dsb = float(p.get("shell_bundle_diametral_clearance_m", 0.006))
    dtb = float(p.get("tube_baffle_hole_diametral_clearance_m", 0.0008))
    rho = float(p["shell_density_kg_m3"])
    mu = float(p["shell_viscosity_Pa_s"])
    cp = float(p["shell_cp_kJ_kgK"]) * 1000.0
    k = float(p["shell_conductivity_W_mK"])
    muw = float(p.get("shell_wall_viscosity_Pa_s", mu))
    L = float(p["tube_length_m"])
    Bin = float(p.get("baffle_spacing_inlet_m", B))
    Bout = float(p.get("baffle_spacing_outlet_m", B))
    Nt = int(p["tube_count"])
    if min(Ds, Do, Pt, B, rho, mu, cp, k, muw, L, Bin, Bout) <= 0:
        raise ValueError("Geometry and fluid properties must be positive.")
    if Pt <= Do or Dsb >= Ds or dtb < 0 or not 0 < cut < 0.5:
        raise ValueError("Review tube pitch, clearances, shell diameter and baffle cut.")
    if str(p.get("tube_layout", "triangular")).lower() not in ("triangular", "square"):
        raise ValueError("tube_layout must be triangular or square.")
    m_shell = float(p["hot_mass_flow_kg_s"] if str(p["tube_side"]).lower()=="cold" else p["cold_mass_flow_kg_s"])
    if m_shell <= 0:
        raise ValueError("Shell-side mass flow must be positive.")

    # Documented geometry quantities.
    Dotl = float(adv.get("bundle_outer_diameter_m") or (Ds - Dsb))
    Dctl = Dotl - Do
    Sm = B * (Ds - Dotl + Dctl * (Pt - Do) / Pt)
    if Sm <= 0:
        raise ValueError("Derived shell crossflow area is non-positive; check geometry inputs.")
    Nb = max(int(np.floor((L - Bin - Bout) / B)) + 1, 1)
    # Window tube fraction estimated from circular segment fraction; replace with layout-derived value if known.
    Aseg_gross = _circle_segment_area_from_height(Ds, cut * Ds)
    Fw = float(np.clip(Aseg_gross / (np.pi * Ds**2 / 4.0), 0.02, 0.48))
    Asb = np.pi * Ds * Dsb / 2.0 * (1.0 - cut)
    Atb = np.pi / 4.0 * ((Do + dtb)**2 - Do**2) * Nt * (1.0 - Fw)
    # Approximate number of tubes in window from tube count times the window fraction.
    Nt_window_est = max(int(round(Nt * Fw)), 0)
    A_tube_blockage_est = Nt_window_est * np.pi * Do**2 / 4.0
    Aw_net_est = max(Aseg_gross - A_tube_blockage_est, 1e-6)
    # Prefer measured/CAD-derived net window free area when supplied and documented.
    Aw_override = adv.get("window_net_free_area_m2")
    Aw_net = float(Aw_override) if Aw_override is not None else Aw_net_est
    if Aw_net <= 0 or Aw_net >= np.pi * Ds**2 / 4.0:
        raise ValueError("Window net free area override must be positive and smaller than shell cross-sectional area.")

    # Row-count estimates from simple shell/pitch geometry; user overrides are preferred.
    Ncl_est = max(int(round(Dctl * (1.0 - cut) / Pt)), 1)
    Ntw_est = max(int(round(2.0 * np.sqrt(Aw_net / np.pi) / Pt)), 1)
    Ncw_est = max(int(round(Dctl * cut / Pt)), 1)
    Ncl = int(adv.get("tube_rows_crossflow_override") or Ncl_est)
    Ntw = int(adv.get("tube_rows_window_override") or Ntw_est)
    Ncw = int(adv.get("tube_rows_endzone_override") or Ncw_est)
    if min(Ncl, Ntw, Ncw) < 1:
        raise ValueError("Tube row counts must be at least 1.")

    Gs = m_shell / Sm
    Re_o = Do * Gs / mu
    Pr = cp * mu / k
    Prw = cp * muw / k
    layout = str(p.get("tube_layout", "triangular")).lower()
    if layout == "triangular":
        Nu = 0.35 * (Pt/Do)**0.2 * Re_o**0.6 * Pr**0.36 * (Pr/Prw)**0.25
        h_ideal = Nu * k / Do
    else:
        Nu = 0.27 * Re_o**0.63 * Pr**0.36 * (Pr/Prw)**0.25
        h_ideal = Nu * k / Do

    # Geometry-based leakage/bypass ratios. Bell–Delaware factor forms are kept visible;
    # exact factors still need actual tube-window fraction and stream-area data.
    r_l = (Asb + Atb) / max(Sm, 1e-12)
    r_s = Asb / max(Asb + Atb, 1e-12)
    Jl = float(np.clip(0.44*(1-r_s) + (1-0.44*(1-r_s))*np.exp(-2.2*r_l), 0.2, 1.0))
    Abypass = np.pi * Ds * Dsb / 2.0
    Fsbp = float(np.clip(Abypass / max(Abypass + Sm, 1e-12), 0.0, 0.95))
    seal_pairs = max(int(p.get("sealing_strip_pairs", 0)), 0)
    ratio_ss = min(2.0*seal_pairs/max(Ncl, 1), 0.5)
    Cbh = 1.35 if Re_o <= 100 else 1.25
    Jb = float(np.exp(-Cbh*Fsbp*(1-ratio_ss**(1/3)))) if ratio_ss < 0.5 else 1.0
    Fc = float(np.clip(1.0 - 2.0*cut, 0.05, 0.95))
    Jc = 0.55 + 0.72*Fc
    n_exp = 0.2 if Re_o > 100 else 1.0
    Js_num = max(Nb-1, 0) + (Bin/B)**(1-n_exp) + (Bout/B)**(1-n_exp)
    Js_den = max(Nb-1, 0) + Bin/B + Bout/B
    Js = float(np.clip(Js_num/max(Js_den, 1e-12), 0.5, 1.2))
    Jr = 1.0 if Re_o >= 100 else float(np.clip((Re_o/100.0)**0.2, 0.2, 1.0))
    h_shell = h_ideal * Jc * Jl * Jb * Js * Jr

    # Published componentized pressure-drop equations; fi is a required source-sensitive input.
    fi = float(adv.get("ideal_bank_friction_factor_fi", 0.03))
    if fi <= 0:
        raise ValueError("ideal_bank_friction_factor_fi must be positive; enter a source-backed value.")
    dp_id = 2.0 * fi * Ncl * Gs**2 / rho
    Rl = Jl
    Rb = Jb
    dp_cross = max(Nb-1, 0) * dp_id * Rl * Rb
    Gw = m_shell / np.sqrt(Sm * Aw_net)
    dp_window = Nb * (2.0 + 0.6*Ntw) * Gw**2 / (2.0*rho) * Rl
    dp_end = 2.0 * dp_id * (1.0 + Ncw/max(Ncl,1)) * Rb
    dp_total = dp_cross + dp_window + dp_end
    Qv = m_shell / rho
    pump_w = dp_total * Qv / float(p["pump_efficiency"])

    flags = []
    if adv.get("bundle_outer_diameter_m") is None: flags.append("Bundle OD derived from shell/baffle clearance; confirm from drawing")
    if adv.get("tube_rows_crossflow_override") is None or adv.get("tube_rows_window_override") is None or adv.get("tube_rows_endzone_override") is None:
        flags.append("At least one tube-row count estimated; replace with actual layout counts")
    if "ideal_bank_friction_factor_fi" in adv and adv["ideal_bank_friction_factor_fi"] == 0.03:
        flags.append("Ideal-bank friction factor fi=0.03 is a placeholder, not a validated Taborek/table value")
    if adv.get("window_net_free_area_m2") is None:
        flags.append("Window free area uses estimated tube blockage; obtain exact window geometry for rating")
    else:
        flags.append("Window net free area supplied from drawing/CAD; verify definition and units")
    flags.append("Single-phase, constant-property calculation; nozzle pressure drop is not included")
    return {
        "Status": "GEOMETRY-AWARE ENGINEERING SCREEN — NOT VALIDATED",
        "Shell_mass_flow_kg_s": m_shell,
        "Bundle_outer_diameter_m": Dotl,
        "Tube_limit_diameter_m": Dctl,
        "Baffle_count_est": Nb,
        "Crossflow_area_Sm_m2": Sm,
        "Shell_to_baffle_leakage_area_Asb_m2": Asb,
        "Tube_to_baffle_leakage_area_Atb_m2": Atb,
        "Window_gross_area_m2": Aseg_gross,
        "Window_net_area_est_m2": Aw_net,
        "Window_tube_fraction_est": Fw,
        "Tube_rows_crossflow_Ncl": Ncl,
        "Tube_rows_window_Ntw": Ntw,
        "Tube_rows_endzone_Ncw": Ncw,
        "Shell_mass_velocity_Gs_kg_m2s": Gs,
        "Shell_Reynolds_number_based_on_Do": Re_o,
        "Shell_Prandtl_number": Pr,
        "Ideal_Nusselt_number": Nu,
        "Ideal_shell_HTC_W_m2K": h_ideal,
        "Jc": float(Jc), "Jl": float(Jl), "Jb": float(Jb), "Js": float(Js), "Jr": float(Jr),
        "Corrected_shell_HTC_W_m2K": float(h_shell),
        "Ideal_bank_friction_factor_fi_input": fi,
        "Ideal_crossflow_DP_per_space_Pa": float(dp_id),
        "Crossflow_DP_Pa": float(dp_cross),
        "Window_DP_Pa": float(dp_window),
        "Endzone_DP_Pa": float(dp_end),
        "Total_shell_DP_kPa_excluding_nozzles": float(dp_total/1000.0),
        "Shell_pumping_power_W_est": float(pump_w),
        "Allowable_shell_DP_kPa": float(p["allowable_shell_dp_kPa"]),
        "Shell_DP_limit_check": "PASS (screen only)" if dp_total/1000.0 <= float(p["allowable_shell_dp_kPa"]) else "FAIL / REVIEW",
        "Estimation_flags": flags
    }

geometry_aware_rating = calculate_geometry_aware_bell_delaware(DESIGN_INPUTS)
display(pd.DataFrame([{
    k: ("; ".join(v) if isinstance(v, list) else v) for k,v in geometry_aware_rating.items()
}]).T.rename(columns={0:"Value"}))

### 28.1 Reference comparison: DWSIM / NeqSim / vendor / experimental data

A model is not “validated” merely because the equations run. Enter reference results for the **same geometry, fluid properties, flow rate, and operating point**. Relative error is calculated only when a reference value is supplied. If no reference results are entered, the status remains `AWAITING_REFERENCE_DATA`.

DWSIM documents the full Bell–Delaware pressure-drop component equations and ideal-bank friction factor as layout/Re-dependent. The reference calculation should use the same assumptions and should include or exclude nozzle losses consistently.

In [ ]:
def compare_against_reference(model_result, adv=None):
    adv = dict(BELL_DELAWARE_ADVANCED if adv is None else adv)
    pairs = [
        ("Corrected_shell_HTC_W_m2K", "reference_h_shell_W_m2K", "h_shell_W_m2K"),
        ("Total_shell_DP_kPa_excluding_nozzles", "reference_dp_shell_kPa", "dp_shell_kPa"),
    ]
    rows = []
    for model_key, ref_key, label in pairs:
        model_value = float(model_result[model_key])
        ref_value = adv.get(ref_key)
        if ref_value is None:
            rows.append({"Metric": label, "Model_value": model_value, "Reference_value": None,
                         "Absolute_error": None, "Relative_error_percent": None,
                         "Status": "AWAITING_REFERENCE_DATA"})
        else:
            ref_value = float(ref_value)
            if ref_value == 0:
                rel = None
                status = "INVALID_REFERENCE_ZERO"
            else:
                rel = abs(model_value-ref_value)/abs(ref_value)*100.0
                status = "WITHIN_USER_TOLERANCE" if rel <= float(adv.get("benchmark_tolerance_percent", 10.0)) else "OUTSIDE_USER_TOLERANCE"
            rows.append({"Metric": label, "Model_value": model_value, "Reference_value": ref_value,
                         "Absolute_error": abs(model_value-ref_value), "Relative_error_percent": rel,
                         "Status": status})
    return pd.DataFrame(rows)

benchmark_df = compare_against_reference(geometry_aware_rating, BELL_DELAWARE_ADVANCED)
print("Reference source:", BELL_DELAWARE_ADVANCED.get("reference_source_label", "Not supplied"))
display(benchmark_df)

# Deterministic software sanity checks. These are unit/sanity tests, NOT physical validation.
assert geometry_aware_rating["Crossflow_area_Sm_m2"] > 0
assert geometry_aware_rating["Window_net_area_est_m2"] > 0
assert geometry_aware_rating["Total_shell_DP_kPa_excluding_nozzles"] > 0
assert abs((geometry_aware_rating["Crossflow_DP_Pa"] + geometry_aware_rating["Window_DP_Pa"] + geometry_aware_rating["Endzone_DP_Pa"])/1000.0 - geometry_aware_rating["Total_shell_DP_kPa_excluding_nozzles"]) < 1e-9
assert 0 < geometry_aware_rating["Corrected_shell_HTC_W_m2K"] <= geometry_aware_rating["Ideal_shell_HTC_W_m2K"] * 1.5
print("Geometry/pressure-drop sanity tests: PASS (software checks only; not engineering validation).")

### Validation checklist before design use

- [ ] Replace all estimated geometry quantities with the actual exchanger drawing: bundle OD, window free area, row counts, sealing strips and clearances.
- [ ] Replace `ideal_bank_friction_factor_fi` with the value from a documented layout/Reynolds correlation or trusted rating package; record the source.
- [ ] Enter matching reference HTC and pressure drop in `BELL_DELAWARE_ADVANCED` and record the source label.
- [ ] Check relative errors over multiple operating points, not only one point; document tolerances and correlation applicability.
- [ ] Compare pressure-drop components with nozzle losses treated consistently.
- [ ] Obtain qualified heat-exchanger engineering review before production decisions.

A comparison harness is provided; no reference values were supplied in this session, so no numerical external validation is claimed.

## 29. Export preliminary design report
The JSON report contains assumptions, inputs, outputs, constraints and explicit limitations. Review the report before sharing or using it.

In [ ]:
report = {
    "title": "Preliminary Shell-and-Tube Heat Exchanger Design Report",
    "status": "PRELIMINARY SCREENING — NOT CODE CERTIFIED",
    "inputs": DESIGN_INPUTS,
    "results": preliminary_result,
    "shell_side_kern_screening": shell_side_result,
    "bell_delaware_inspired_screening": bell_delaware_screening,
    "shell_side_correlation_based_rating": shell_side_correlation_rating,
    "geometry_aware_bell_delaware_rating": geometry_aware_rating,
    "reference_benchmark_comparison": benchmark_df.replace({np.nan: None}).to_dict(orient="records"),
    "benchmark_reference_source": BELL_DELAWARE_ADVANCED.get("reference_source_label", "Not supplied"),
    "geometry_candidates_top_20": (
        geometry_candidates.head(20).replace({np.nan: None}).to_dict(orient="records")
        if not geometry_candidates.empty else []
    ),
    "limitations": [
        "The Bell–Delaware-inspired factors are retained for comparison; see shell_side_correlation_based_rating for the newer correlation-based screening calculation.",
        "The shell-side calculations are not independently validated against vendor software, experimental data or a benchmark design.",
        "Geometry-aware Bell–Delaware section estimates tube rows and window free area unless user overrides are supplied.",
        "Ideal tube-bank friction factor defaults to a placeholder and must be replaced with a source-backed layout/Reynolds value.",
        "Reference comparison is only meaningful when matching DWSIM/NeqSim/vendor or experimental results are entered.",
        "No TEMA compliance certification performed.",
        "No ASME pressure-boundary or tubesheet mechanical design performed.",
        "No tube vibration, nozzle, impingement, thermal expansion or gasket/bolting checks performed.",
        "Constant fluid properties and simplified fouling-resistance increment are used.",
        "Dittus-Boelter is only a screening correlation for suitable turbulent flow.",
        "Leakage/bypass areas and pressure-drop components use simplified geometry proxies and are not a complete exact Bell–Delaware pressure-drop implementation.",
        "Baffle-window, nozzle, pass-partition, bundle geometry, sealing-strip details and flow distribution need validated methods/vendor data.",
        "Pressure-drop estimates require validation against real geometry and approved correlations.",
        "Use an independent reference implementation or vendor rating and documented error tolerances before any production use."
    ]
}
with open("shell_tube_preliminary_design_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=lambda x: None if isinstance(x, (np.floating, np.integer)) and not np.isfinite(x) else float(x) if isinstance(x, (np.floating, np.integer)) else str(x))
geometry_candidates.to_csv("shell_tube_geometry_candidates.csv", index=False)
pd.DataFrame([{
    k: v for k, v in preliminary_result.items() if not isinstance(v, dict)
}]).to_csv("shell_tube_preliminary_design_summary.csv", index=False)
print("Exported:")
print("- shell_tube_preliminary_design_report.json")
print("- shell_tube_geometry_candidates.csv")
print("- shell_tube_preliminary_design_summary.csv")
with open("shell_side_kern_screening.json", "w", encoding="utf-8") as f:
    json.dump(shell_side_result, f, indent=2, ensure_ascii=False)
print("- shell_side_kern_screening.json")
with open("shell_side_bell_delaware_inspired_screening.json", "w", encoding="utf-8") as f:
    json.dump(bell_delaware_screening, f, indent=2, ensure_ascii=False, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else str(x))
print("- shell_side_bell_delaware_inspired_screening.json")
with open("shell_side_correlation_based_rating.json", "w", encoding="utf-8") as f:
    json.dump(shell_side_correlation_rating, f, indent=2, ensure_ascii=False, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else str(x))
print("- shell_side_correlation_based_rating.json")


## 30. Independent validation protocol and engineering release gate

**Important status:** this notebook cannot itself appoint or certify a qualified engineer, and no external benchmark dataset or signed engineering review was supplied in this session. This section prepares a repeatable validation package and deliberately prevents the tool from claiming validation until evidence is entered.

The independent validation process has three separate gates:

1. **Software verification** — deterministic unit/sanity tests and unit consistency.
2. **Thermal-hydraulic validation** — compare multiple matched operating points against an independent implementation (e.g., DWSIM/NeqSim/vendor rating) or measured test data. Match geometry, fluid properties, flow rates, nozzle-loss treatment and correlation applicability.
3. **Engineering approval** — qualified heat-exchanger engineer reviews the design basis, TEMA edition/class, pressure-boundary code basis, materials, vibration, thermal expansion, fabrication tolerances and test requirements; signs and dates the review record.

A passing software test is not physical validation. A comparison to a second code is also not fully independent if both implementations share the same assumptions or source correlations. See the official TEMA standards page for the current edition and the applicable ASME Section VIII requirements for the pressure-boundary design basis. Sources: https://tema.org/standards/ and https://www.asme.org/codes-standards/find-codes-standards/bpvc-section-viii-rules-construction-pressure-vessels.

### 30.1 Benchmark input template

Use one row per operating point. Populate reference values only from a traceable source and ensure the model and reference use the same geometry and boundary conditions. Do not mark a case as matched merely because it is a similar exchanger.

In [ ]:
# Create a blank CSV template for independent benchmark evidence.
VALIDATION_TEMPLATE_COLUMNS = [
    "case_id", "reference_source", "source_version_or_date",
    "geometry_matched_yes_no", "fluid_properties_matched_yes_no",
    "operating_point_matched_yes_no", "nozzle_losses_treatment_matched_yes_no",
    "reference_h_shell_W_m2K", "model_h_shell_W_m2K",
    "reference_dp_shell_kPa", "model_dp_shell_kPa",
    "h_tolerance_percent", "dp_tolerance_percent",
    "reviewer_name", "reviewer_qualification", "review_date_YYYY_MM_DD",
    "comments"
]
validation_template = pd.DataFrame(columns=VALIDATION_TEMPLATE_COLUMNS)
validation_template.to_csv("independent_validation_benchmark_template.csv", index=False)
print("Created independent_validation_benchmark_template.csv")
print("Populate with traceable reference data; blank template is not validation evidence.")

### 30.2 Calculate errors and apply evidence checks

Tolerances are user/project-defined; there is no universal acceptance tolerance suitable for every exchanger. The gate below requires a minimum number of fully matched operating points and checks both heat-transfer coefficient and shell-side pressure drop. It reports evidence readiness, not code compliance.

In [ ]:
def evaluate_independent_benchmarks(benchmarks, min_matched_cases=3):
    """Evaluate traceable, matched benchmark rows; this is not a certification function.

    Required columns are defined in VALIDATION_TEMPLATE_COLUMNS. Acceptance tolerances
    must be supplied per row and agreed in the project validation plan.
    """
    required = set(VALIDATION_TEMPLATE_COLUMNS)
    missing = sorted(required - set(benchmarks.columns))
    if missing:
        raise ValueError(f"Benchmark table missing required columns: {missing}")
    dfb = benchmarks.copy()
    flag_cols = ["geometry_matched_yes_no", "fluid_properties_matched_yes_no",
                 "operating_point_matched_yes_no", "nozzle_losses_treatment_matched_yes_no"]
    for col in flag_cols:
        dfb[col] = dfb[col].astype(str).str.strip().str.lower()
    numeric_cols = ["reference_h_shell_W_m2K", "model_h_shell_W_m2K",
                    "reference_dp_shell_kPa", "model_dp_shell_kPa",
                    "h_tolerance_percent", "dp_tolerance_percent"]
    for col in numeric_cols:
        dfb[col] = pd.to_numeric(dfb[col], errors="coerce")
    dfb["matched_case"] = dfb[flag_cols].eq("yes").all(axis=1)
    dfb["h_error_percent"] = np.where(
        (dfb["reference_h_shell_W_m2K"] > 0) & dfb["model_h_shell_W_m2K"].notna(),
        abs(dfb["model_h_shell_W_m2K"] - dfb["reference_h_shell_W_m2K"]) /
        dfb["reference_h_shell_W_m2K"] * 100.0, np.nan)
    dfb["dp_error_percent"] = np.where(
        (dfb["reference_dp_shell_kPa"] > 0) & dfb["model_dp_shell_kPa"].notna(),
        abs(dfb["model_dp_shell_kPa"] - dfb["reference_dp_shell_kPa"]) /
        dfb["reference_dp_shell_kPa"] * 100.0, np.nan)
    dfb["row_evidence_complete"] = (
        dfb["matched_case"] & dfb["reference_source"].fillna("").astype(str).str.strip().ne("") &
        dfb["source_version_or_date"].fillna("").astype(str).str.strip().ne("") &
        dfb["reviewer_name"].fillna("").astype(str).str.strip().ne("") &
        dfb["reviewer_qualification"].fillna("").astype(str).str.strip().ne("") &
        dfb["review_date_YYYY_MM_DD"].fillna("").astype(str).str.strip().ne("") &
        dfb["h_error_percent"].notna() & dfb["dp_error_percent"].notna() &
        dfb["h_tolerance_percent"].gt(0) & dfb["dp_tolerance_percent"].gt(0)
    )
    dfb["row_within_tolerance"] = (
        dfb["row_evidence_complete"] &
        dfb["h_error_percent"].le(dfb["h_tolerance_percent"]) &
        dfb["dp_error_percent"].le(dfb["dp_tolerance_percent"])
    )
    matched = int(dfb["matched_case"].sum())
    complete = int(dfb["row_evidence_complete"].sum())
    all_pass = bool(complete >= min_matched_cases and dfb.loc[dfb["row_evidence_complete"], "row_within_tolerance"].all())
    summary = {
        "matched_cases": matched,
        "complete_evidence_cases": complete,
        "minimum_required_cases": int(min_matched_cases),
        "benchmark_gate": "PASS — benchmark criteria met; engineering sign-off still required" if all_pass else "BLOCKED — insufficient or out-of-tolerance evidence",
        "is_independent_physical_validation": False,
        "note": "Review source independence, model assumptions, test uncertainty and project-specific tolerances before accepting this result."
    }
    return dfb, summary

# If you have populated the CSV template, load it and run:
# benchmark_data = pd.read_csv("independent_validation_benchmark_template.csv")
# benchmark_detail, benchmark_summary = evaluate_independent_benchmarks(benchmark_data, min_matched_cases=3)
# display(benchmark_detail)
# print(json.dumps(benchmark_summary, indent=2))
print("Benchmark evaluator ready. No external reference data were provided, so no physical validation is claimed.")

### 30.3 Engineer review and release checklist

The reviewer should be a qualified engineer experienced in shell-and-tube exchanger thermal design and pressure-vessel requirements. This notebook does not perform or attest to that review. Confirm the project-adopted editions of TEMA and the applicable construction code (for example, ASME BPVC Section VIII where specified), rather than assuming a code edition.

- [ ] Design basis approved: service, duty, operating/design pressure and temperature, upset cases, allowable pressure drops, fouling and corrosion allowances.
- [ ] Fluid property source and evaluation temperatures documented; phase change, viscosity correction and non-Newtonian behaviour assessed where relevant.
- [ ] As-built geometry verified: TEMA type, tube layout and counts, baffle cut/spacing, window free area, bundle diameter, all leakage clearances, sealing strips, pass lanes and nozzles.
- [ ] Bell–Delaware correlations and ideal-bank heat-transfer/friction-factor sources recorded with applicability ranges.
- [ ] At least three matched operating points benchmarked (or a stricter project-defined test matrix), including measurement/reference uncertainty and approved tolerances.
- [ ] Heat balance, outlet temperatures, overall U, fouling, both-side pressure drops and pumping power reconciled.
- [ ] Mechanical checks completed by responsible engineer: pressure boundary, tubesheet, channel/shell, flanges/bolting/gaskets, supports, thermal expansion, nozzle loads, impingement and vibration.
- [ ] Materials, corrosion/erosion, cleaning access, inspection, fabrication tolerances, hydrotest/leak test and applicable statutory requirements reviewed.
- [ ] Deviations, assumptions, software version, inputs, outputs and reviewer comments archived.
- [ ] Qualified engineer signs and dates the release record.

**Release rule:** until benchmark evidence and qualified sign-off are completed, label every exported report `PRELIMINARY — NOT APPROVED FOR FABRICATION OR GUARANTEED PERFORMANCE`.

In [ ]:
# Machine-readable review/sign-off record. Fill only after the actual review has occurred.
engineering_review_record = {
    "project_id": "",
    "exchanger_tag": "",
    "design_basis_document": "",
    "tema_edition_and_class": "",
    "construction_code_and_edition": "",
    "independent_reference_source_and_version": "",
    "matched_benchmark_cases": 0,
    "benchmark_tolerance_approved_by_project": False,
    "mechanical_design_review_complete": False,
    "thermal_hydraulic_review_complete": False,
    "vibration_and_thermal_expansion_review_complete": False,
    "qualified_reviewer_name": "",
    "reviewer_qualification_or_registration": "",
    "review_date": "",
    "open_actions": ["No external validation evidence or engineer sign-off supplied yet"],
    "release_status": "BLOCKED — PRELIMINARY ONLY"
}
with open("engineering_review_signoff_template.json", "w", encoding="utf-8") as f:
    json.dump(engineering_review_record, f, indent=2, ensure_ascii=False)
print("Created engineering_review_signoff_template.json")
print("Release remains BLOCKED until evidence and qualified sign-off are genuinely completed.")

### 30.4 Standards and independent references

- TEMA standards and current edition information: https://tema.org/standards/
- NeqSim Bell–Delaware method documentation: https://equinor.github.io/neqsimhome/javadoc/site/apidocs/neqsim/process/mechanicaldesign/heatexchanger/BellDelawareMethod.html
- Gonçalves et al., Bell–Delaware design/optimization paper: https://www.ou.edu/class/che-design/pub-papers/Linear%20method%20for%20the%20design%20and%20optimization%20of%20shell%20and%20tube%20heat%20exchangers%20using%20the%20Bell-Delaware%20Method%28Goncalves%20et%20al%29-19.pdf
- Published comparisons and case studies can support a benchmark plan, but only a geometry/property/operating-point-matched case is suitable for a quantitative comparison.

**Current release status for this notebook: BLOCKED / PRELIMINARY.** This session created a validation workflow and review templates; it did not obtain external rating outputs, physical test data, or an actual qualified engineer's signed review.

In [ ]:
# Final release status — deliberately blocked until actual benchmark evidence and sign-off exist.
final_release_status = {
    "status": "BLOCKED — PRELIMINARY ONLY",
    "independent_benchmark_data_provided": False,
    "qualified_engineer_signoff_provided": False,
    "not_for_fabrication_or_guaranteed_performance": True,
    "required_next_actions": [
        "Run matched operating points against a traceable independent reference or measured data",
        "Review model applicability, uncertainties and project-approved tolerances",
        "Complete mechanical/code review and obtain qualified engineer sign-off"
    ]
}
if "report" in globals() and isinstance(report, dict):
    report["independent_validation_and_engineering_release_gate"] = final_release_status
    report["status"] = final_release_status["status"]
    with open("shell_tube_preliminary_design_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=str)
with open("engineering_release_status.json", "w", encoding="utf-8") as f:
    json.dump(final_release_status, f, indent=2, ensure_ascii=False)
print(json.dumps(final_release_status, indent=2, ensure_ascii=False))


# 31. Professional engineering workflow pack

This section adds a concise executive summary, input/data-quality gates, sensitivity screening, uncertainty ranges, and a design-risk register. These are **decision-support tools**, not independent physical validation or engineering approval.

**Important:** sensitivity ranges below are illustrative defaults. Replace them with drawing tolerances, measured property uncertainty, and project-approved ranges before relying on the results.


In [ ]:
# 31.1 Executive summary and data-quality gates
import math
from datetime import datetime, timezone

def build_professional_summary(inputs, rating, benchmark_comparison=None):
    required = [
        "hot_mass_flow_kg_s", "cold_mass_flow_kg_s", "tube_inner_diameter_m",
        "tube_outer_diameter_m", "tube_length_m", "tube_count", "shell_inside_diameter_m",
        "baffle_spacing_m", "tube_pitch_m", "shell_density_kg_m3",
        "shell_viscosity_Pa_s", "shell_cp_kJ_kgK", "shell_conductivity_W_mK"
    ]
    missing = [k for k in required if k not in inputs or inputs[k] is None]
    errors = []
    warnings = []
    if not missing:
        if inputs["tube_pitch_m"] <= inputs["tube_outer_diameter_m"]:
            errors.append("Tube pitch must be greater than tube outside diameter.")
        if inputs["tube_outer_diameter_m"] <= inputs["tube_inner_diameter_m"]:
            errors.append("Tube OD must exceed tube ID.")
        if inputs["baffle_spacing_m"] >= inputs["shell_inside_diameter_m"]:
            warnings.append("Baffle spacing is at least shell ID; confirm design applicability.")
        if inputs.get("tube_layout", "").lower() not in ("triangular", "square"):
            errors.append("Tube layout must be triangular or square.")
    else:
        errors.append("Required inputs missing: " + ", ".join(missing))

    h = rating.get("Corrected_shell_HTC_W_m2K")
    dp = rating.get("Total_shell_DP_kPa_excluding_nozzles")
    if h is None or not np.isfinite(float(h)) or float(h) <= 0:
        errors.append("Corrected shell-side HTC is missing or non-positive.")
    if dp is None or not np.isfinite(float(dp)) or float(dp) <= 0:
        errors.append("Shell-side pressure drop is missing or non-positive.")
    if rating.get("Estimation_flags"):
        warnings.extend([str(x) for x in rating["Estimation_flags"]])
    benchmark_status = "NOT PROVIDED"
    if benchmark_comparison is not None and len(benchmark_comparison):
        if "Status" in benchmark_comparison.columns:
            benchmark_status = "; ".join(sorted(set(benchmark_comparison["Status"].astype(str))))
    return {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "tool_status": "PRELIMINARY ENGINEERING SCREENING",
        "release_status": "BLOCKED — NOT FOR FABRICATION",
        "shell_side_htc_W_m2K": float(h) if h is not None else None,
        "shell_side_dp_kPa_excluding_nozzles": float(dp) if dp is not None else None,
        "shell_dp_allowable_kPa": float(inputs.get("allowable_shell_dp_kPa", np.nan)),
        "shell_dp_limit_screen": (
            "WITHIN INPUT LIMIT (SCREEN ONLY)"
            if dp is not None and float(dp) <= float(inputs.get("allowable_shell_dp_kPa", -1))
            else "EXCEEDS LIMIT OR LIMIT NOT VALID"
        ),
        "benchmark_status": benchmark_status,
        "input_errors": errors,
        "review_warnings": warnings,
        "independent_validation_complete": False,
        "qualified_engineer_signoff_complete": False
    }

professional_summary = build_professional_summary(
    DESIGN_INPUTS, geometry_aware_rating, benchmark_df
)
print(json.dumps(professional_summary, indent=2, ensure_ascii=False))


## 31.2 Sensitivity screening

This one-at-a-time analysis varies selected parameters while holding others fixed. It helps identify influential inputs; it does **not** quantify a statistically validated uncertainty distribution. Change the ranges to match the real design basis.


In [ ]:
# 31.2 One-at-a-time sensitivity analysis (illustrative ranges)
SENSITIVITY_RANGES = {
    "cold_mass_flow_kg_s": (0.90, 1.10),
    "baffle_spacing_m": (0.90, 1.10),
    "shell_bundle_diametral_clearance_m": (0.75, 1.25),
    "tube_baffle_hole_diametral_clearance_m": (0.75, 1.25),
    "shell_viscosity_Pa_s": (0.90, 1.10),
    "shell_conductivity_W_mK": (0.95, 1.05),
}
def run_one_at_a_time_sensitivity(base_inputs, ranges=None):
    ranges = SENSITIVITY_RANGES if ranges is None else ranges
    baseline = calculate_geometry_aware_bell_delaware(dict(base_inputs))
    baseline_h = float(baseline["Corrected_shell_HTC_W_m2K"])
    baseline_dp = float(baseline["Total_shell_DP_kPa_excluding_nozzles"])
    records = []
    for key, (low_mult, high_mult) in ranges.items():
        if key not in base_inputs:
            continue
        for label, mult in (("low", low_mult), ("high", high_mult)):
            trial = dict(base_inputs)
            trial[key] = float(base_inputs[key]) * float(mult)
            try:
                result = calculate_geometry_aware_bell_delaware(trial)
                h = float(result["Corrected_shell_HTC_W_m2K"])
                dp = float(result["Total_shell_DP_kPa_excluding_nozzles"])
                records.append({
                    "parameter": key, "case": label, "multiplier": mult,
                    "input_value": trial[key],
                    "HTC_change_percent": (h / baseline_h - 1) * 100 if baseline_h else np.nan,
                    "DP_change_percent": (dp / baseline_dp - 1) * 100 if baseline_dp else np.nan,
                    "status": "SCREENING ONLY"
                })
            except Exception as exc:
                records.append({"parameter": key, "case": label, "multiplier": mult,
                                "input_value": trial[key], "HTC_change_percent": np.nan,
                                "DP_change_percent": np.nan, "status": f"ERROR: {exc}"})
    return pd.DataFrame(records)

sensitivity_results = run_one_at_a_time_sensitivity(DESIGN_INPUTS)
display(sensitivity_results)
if not sensitivity_results.empty:
    sensitivity_results.to_csv("heat_exchanger_sensitivity_screening.csv", index=False)
    sensitivity_rank = (sensitivity_results[sensitivity_results["status"] == "SCREENING ONLY"]
        .groupby("parameter")[["HTC_change_percent", "DP_change_percent"]]
        .apply(lambda g: g.abs().max()).sort_values("DP_change_percent", ascending=False))
    print("Largest tested pressure-drop sensitivities (absolute % change):")
    display(sensitivity_rank)


## 31.3 Uncertainty envelope

The Monte Carlo envelope below uses explicitly configurable uniform perturbations around selected inputs. The ranges are illustrative only. The output is a **scenario envelope**, not a probability-certified confidence interval. Do not use it for guarantees or safety factors without an approved uncertainty method.


In [ ]:
# 31.3 Scenario envelope / Monte Carlo-style screening
UNCERTAINTY_RELATIVE_BOUNDS = {
    "cold_mass_flow_kg_s": 0.05,
    "baffle_spacing_m": 0.03,
    "shell_viscosity_Pa_s": 0.10,
    "shell_conductivity_W_mK": 0.05,
    "shell_bundle_diametral_clearance_m": 0.20,
    "tube_baffle_hole_diametral_clearance_m": 0.20,
}
def run_uncertainty_envelope(base_inputs, n_cases=250, seed=42, relative_bounds=None):
    bounds = UNCERTAINTY_RELATIVE_BOUNDS if relative_bounds is None else relative_bounds
    rng = np.random.default_rng(seed)
    records = []
    for case_id in range(int(n_cases)):
        trial = dict(base_inputs)
        for key, rel in bounds.items():
            if key in trial:
                trial[key] = float(trial[key]) * float(rng.uniform(1-rel, 1+rel))
        try:
            out = calculate_geometry_aware_bell_delaware(trial)
            records.append({
                "case_id": case_id + 1,
                "shell_htc_W_m2K": float(out["Corrected_shell_HTC_W_m2K"]),
                "shell_dp_kPa_excluding_nozzles": float(out["Total_shell_DP_kPa_excluding_nozzles"]),
                "status": "SCREENING ONLY"
            })
        except Exception as exc:
            records.append({"case_id": case_id + 1, "shell_htc_W_m2K": np.nan,
                            "shell_dp_kPa_excluding_nozzles": np.nan, "status": f"ERROR: {exc}"})
    raw = pd.DataFrame(records)
    valid = raw.dropna(subset=["shell_htc_W_m2K", "shell_dp_kPa_excluding_nozzles"])
    summary = pd.DataFrame([
        {"metric": "Shell HTC (W/m²·K)", "p05": valid["shell_htc_W_m2K"].quantile(.05),
         "median": valid["shell_htc_W_m2K"].median(), "p95": valid["shell_htc_W_m2K"].quantile(.95)},
        {"metric": "Shell ΔP excluding nozzles (kPa)", "p05": valid["shell_dp_kPa_excluding_nozzles"].quantile(.05),
         "median": valid["shell_dp_kPa_excluding_nozzles"].median(), "p95": valid["shell_dp_kPa_excluding_nozzles"].quantile(.95)}
    ])
    return raw, summary

uncertainty_cases, uncertainty_summary = run_uncertainty_envelope(DESIGN_INPUTS)
display(uncertainty_summary)
uncertainty_cases.to_csv("heat_exchanger_uncertainty_scenarios.csv", index=False)
uncertainty_summary.to_csv("heat_exchanger_uncertainty_summary.csv", index=False)
print(f"Valid scenario cases: {uncertainty_cases['shell_htc_W_m2K'].notna().sum()} / {len(uncertainty_cases)}")


## 31.4 Design risk register and deliverable checklist

Resolve every high-risk item before design release. This register is a template and does not imply that any review has occurred.


In [ ]:
# 31.4 Risk register — update owners and evidence on the actual project
design_risk_register = pd.DataFrame([
    {"ID":"R-01","Risk":"Bell–Delaware implementation is not independently benchmarked","Severity":"HIGH","Mitigation":"Compare matched cases against a traceable vendor/reference implementation or measured data","Evidence":"","Owner":"","Status":"OPEN"},
    {"ID":"R-02","Risk":"Geometry inputs are approximated rather than drawing-derived","Severity":"HIGH","Mitigation":"Verify tube layout, baffle windows, clearances, tube rows, bundle diameter and sealing strips","Evidence":"","Owner":"","Status":"OPEN"},
    {"ID":"R-03","Risk":"Ideal tube-bank friction factor may be a placeholder","Severity":"HIGH","Mitigation":"Replace with documented layout/Re correlation or reference value and cite source/version","Evidence":"","Owner":"","Status":"OPEN"},
    {"ID":"R-04","Risk":"Nozzle and entrance/exit losses may be excluded","Severity":"MEDIUM","Mitigation":"Add nozzle and local losses using actual nozzle geometry and agreed method","Evidence":"","Owner":"","Status":"OPEN"},
    {"ID":"R-05","Risk":"Mechanical integrity, vibration and thermal expansion not assessed","Severity":"HIGH","Mitigation":"Complete applicable construction-code and qualified engineering checks","Evidence":"","Owner":"","Status":"OPEN"},
    {"ID":"R-06","Risk":"Fluid properties and fouling assumptions may not match operating envelope","Severity":"HIGH","Mitigation":"Use property data at relevant temperatures and documented fouling basis","Evidence":"","Owner":"","Status":"OPEN"},
])
display(design_risk_register)
design_risk_register.to_csv("heat_exchanger_design_risk_register.csv", index=False)


## 31.5 Professional release gate

The tool deliberately remains blocked for fabrication until independent evidence and qualified review are actually recorded. Passing Python tests, producing plausible values, or completing a sensitivity analysis is not proof of engineering validity.


In [ ]:
professional_release_gate = {
    "release_status": "BLOCKED — PRELIMINARY ONLY",
    "python_syntax_and_smoke_checks": "SOFTWARE CHECKS ONLY",
    "independent_benchmark_evidence": "NOT PROVIDED",
    "benchmark_tolerance_approved": False,
    "geometry_verified_against_issued_drawing": False,
    "fluid_properties_and_fouling_basis_verified": False,
    "thermal_hydraulic_review_signed": False,
    "mechanical_code_review_signed": False,
    "vibration_and_thermal_expansion_review_signed": False,
    "qualified_engineer_signoff": False,
    "not_for_fabrication_or_performance_guarantee": True,
    "required_evidence": [
        "Issued exchanger drawing and approved design basis",
        "Traceable independent rating or experimental benchmark cases",
        "Project-approved acceptance tolerances and documented error analysis",
        "Applicable TEMA / construction-code edition confirmed by responsible engineer",
        "Signed thermal, hydraulic, mechanical, vibration and materials review"
    ]
}
with open("heat_exchanger_professional_summary.json", "w", encoding="utf-8") as f:
    json.dump(professional_summary, f, indent=2, ensure_ascii=False, default=str)
with open("heat_exchanger_professional_release_gate.json", "w", encoding="utf-8") as f:
    json.dump(professional_release_gate, f, indent=2, ensure_ascii=False)
with open("heat_exchanger_professional_review.md", "w", encoding="utf-8") as f:
    f.write("# Heat Exchanger Professional Review Pack\\n\\n")
    f.write("**Status: BLOCKED — PRELIMINARY ONLY; not for fabrication.**\\n\\n")
    f.write("## Summary\\n\\n")
    f.write(f"- Shell-side HTC: {professional_summary['shell_side_htc_W_m2K']:.3f} W/m²·K\\n")
    f.write(f"- Shell-side pressure drop excluding nozzles: {professional_summary['shell_side_dp_kPa_excluding_nozzles']:.4f} kPa\\n")
    f.write(f"- Benchmark status: {professional_summary['benchmark_status']}\\n\\n")
    f.write("## Open risks\\n\\n")
    for _, row in design_risk_register.iterrows():
        f.write(f"- **{row['ID']} ({row['Severity']})** {row['Risk']} — {row['Mitigation']}\\n")
    f.write("\\n## Release gate\\n\\n")
    f.write("Independent benchmark evidence and qualified engineer sign-off have not been provided.\\n")
print("Professional review pack exported. Release remains BLOCKED — PRELIMINARY ONLY.")


# 32. Drawing-based geometry import and trusted-reference comparison

This section creates a reproducible workflow for **transcribing dimensions from an approved drawing** and comparing model outputs with a traceable vendor / reference-rating dataset. It does not read CAD drawings automatically: dimensions must be transcribed and independently checked. Keep the drawing number, revision, units, source and reviewer in the input file.

**Release rule:** if actual drawing data or a traceable reference rating are missing, results remain `BLOCKED — PRELIMINARY ONLY`. A matching software output alone is not proof of physical validity.

## 32.1 Create / edit the drawing geometry input file

The template is saved as `heat_exchanger_actual_drawing_geometry.json`. Replace example values with values transcribed from the approved drawing. Use SI units only. `null` means unknown; do not substitute a guess without labelling it as an estimate.

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np

DRAWING_GEOMETRY_TEMPLATE = {
    "drawing_number": "ENTER_DRAWING_NUMBER",
    "drawing_revision": "ENTER_REVISION",
    "drawing_date": "YYYY-MM-DD",
    "transcribed_by": "ENTER_NAME",
    "independently_checked_by": "ENTER_CHECKER",
    "units": "SI",
    "geometry_source_status": "UNVERIFIED_TEMPLATE",
    "shell_inside_diameter_m": None,
    "bundle_outer_diameter_m": None,
    "tube_outer_diameter_m": None,
    "tube_inner_diameter_m": None,
    "tube_length_m": None,
    "tube_count": None,
    "tube_passes": None,
    "shell_passes": None,
    "tube_pitch_m": None,
    "tube_layout": None,
    "baffle_spacing_central_m": None,
    "baffle_spacing_inlet_m": None,
    "baffle_spacing_outlet_m": None,
    "baffle_cut_fraction": None,
    "shell_bundle_diametral_clearance_m": None,
    "tube_baffle_hole_diametral_clearance_m": None,
    "sealing_strip_pairs": None,
    "tube_rows_crossflow": None,
    "tube_rows_window": None,
    "tube_rows_endzone": None,
    "window_net_free_area_m2": None,
    "ideal_bank_friction_factor_fi": None,
    "notes": "Replace all required nulls from the approved drawing; do not use this blank template for design release."
}
geometry_template_path = Path('heat_exchanger_actual_drawing_geometry.json')
if not geometry_template_path.exists():
    geometry_template_path.write_text(json.dumps(DRAWING_GEOMETRY_TEMPLATE, indent=2), encoding='utf-8')
print('Drawing geometry template:', geometry_template_path.resolve())
print('Template status:', DRAWING_GEOMETRY_TEMPLATE['geometry_source_status'])

## 32.2 Load geometry only when drawing provenance and required fields are complete

This loader refuses placeholder or incomplete geometry. It maps only supported geometry fields into the existing design model. Row counts, measured window free area and friction factor are forwarded as explicit overrides when supplied.

In [ ]:
DRAWING_TO_DESIGN_MAP = {
    "shell_inside_diameter_m": "shell_inside_diameter_m",
    "tube_outer_diameter_m": "tube_outer_diameter_m",
    "tube_inner_diameter_m": "tube_inner_diameter_m",
    "tube_length_m": "tube_length_m",
    "tube_count": "tube_count",
    "tube_passes": "tube_passes",
    "shell_passes": "shell_passes",
    "tube_pitch_m": "tube_pitch_m",
    "tube_layout": "tube_layout",
    "baffle_cut_fraction": "baffle_cut_fraction",
    "shell_bundle_diametral_clearance_m": "shell_bundle_diametral_clearance_m",
    "tube_baffle_hole_diametral_clearance_m": "tube_baffle_hole_diametral_clearance_m",
    "baffle_spacing_inlet_m": "baffle_spacing_inlet_m",
    "baffle_spacing_outlet_m": "baffle_spacing_outlet_m",
}
REQUIRED_DRAWING_FIELDS = [
    "drawing_number", "drawing_revision", "transcribed_by", "independently_checked_by",
    "shell_inside_diameter_m", "bundle_outer_diameter_m", "tube_outer_diameter_m",
    "tube_inner_diameter_m", "tube_length_m", "tube_count", "tube_passes",
    "tube_pitch_m", "tube_layout", "baffle_spacing_central_m", "baffle_cut_fraction",
    "shell_bundle_diametral_clearance_m", "tube_baffle_hole_diametral_clearance_m"
]

def load_and_apply_drawing_geometry(path='heat_exchanger_actual_drawing_geometry.json'):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    missing = [k for k in REQUIRED_DRAWING_FIELDS if data.get(k) in (None, '', 'ENTER_DRAWING_NUMBER', 'ENTER_REVISION', 'ENTER_NAME', 'ENTER_CHECKER')]
    if data.get('geometry_source_status') != 'DRAWING_TRANSCRIBED_AND_CHECKED':
        missing.append('geometry_source_status must be DRAWING_TRANSCRIBED_AND_CHECKED after verification')
    if str(data.get('drawing_date', '')).strip() in ('', 'YYYY-MM-DD'):
        missing.append('drawing_date')
    if data.get('units') != 'SI':
        missing.append('units must be SI')
    if missing:
        return data, {'status':'BLOCKED — DRAWING INPUT INCOMPLETE', 'missing_or_unverified':missing}
    numeric_positive = [k for k in REQUIRED_DRAWING_FIELDS if k.endswith('_m')]
    for key in numeric_positive:
        if float(data[key]) <= 0:
            raise ValueError(f'{key} must be positive')
    for key in ('tube_count', 'tube_passes'):
        if int(data[key]) != float(data[key]) or int(data[key]) <= 0:
            raise ValueError(f'{key} must be a positive integer')
    if float(data['bundle_outer_diameter_m']) >= float(data['shell_inside_diameter_m']):
        raise ValueError('Bundle OD must be smaller than shell inside diameter')
    if float(data['tube_pitch_m']) <= float(data['tube_outer_diameter_m']):
        raise ValueError('Tube pitch must exceed tube outside diameter')
    if float(data['tube_inner_diameter_m']) >= float(data['tube_outer_diameter_m']):
        raise ValueError('Tube ID must be smaller than tube OD')
    if not 0 < float(data['baffle_cut_fraction']) < 0.5:
        raise ValueError('Baffle cut fraction must be between 0 and 0.5')
    for source_key, target_key in DRAWING_TO_DESIGN_MAP.items():
        if data.get(source_key) is not None:
            DESIGN_INPUTS[target_key] = data[source_key]
    DESIGN_INPUTS['baffle_spacing_m'] = float(data['baffle_spacing_central_m'])
    if data.get('bundle_outer_diameter_m') is not None:
        BELL_DELAWARE_ADVANCED['bundle_outer_diameter_m'] = float(data['bundle_outer_diameter_m'])
    for source_key, target_key in [
        ('tube_rows_crossflow','tube_rows_crossflow_override'),
        ('tube_rows_window','tube_rows_window_override'),
        ('tube_rows_endzone','tube_rows_endzone_override'),
        ('ideal_bank_friction_factor_fi','ideal_bank_friction_factor_fi')]:
        if data.get(source_key) is not None:
            BELL_DELAWARE_ADVANCED[target_key] = float(data[source_key])
    if data.get('window_net_free_area_m2') is not None:
        BELL_DELAWARE_ADVANCED['window_net_free_area_m2'] = float(data['window_net_free_area_m2'])
    return data, {'status':'DRAWING INPUTS LOADED — REVIEW MODEL APPLICABILITY', 'missing_or_unverified':[]}

# Safe default: no drawing file is assumed to be verified.
if Path('heat_exchanger_actual_drawing_geometry.json').exists():
    drawing_geometry, drawing_gate = load_and_apply_drawing_geometry()
else:
    drawing_geometry, drawing_gate = {}, {'status':'BLOCKED — DRAWING FILE MISSING', 'missing_or_unverified':['geometry file not found']}
print(json.dumps(drawing_gate, indent=2))

## 32.3 Recalculate with the loaded geometry

When the geometry gate is blocked, this section deliberately retains the existing illustrative results and labels them as such. When verified geometry is loaded, the shell-side functions are recalculated from the updated inputs. Process conditions and fluid properties still need to match the actual operating case.

In [ ]:
if drawing_gate['status'].startswith('DRAWING INPUTS LOADED'):
    validate_design_inputs(DESIGN_INPUTS)
    preliminary_result = calculate_shell_tube_preliminary(DESIGN_INPUTS)
    shell_side_result = calculate_shell_side_kern_screening(DESIGN_INPUTS)
    bell_delaware_screening = calculate_bell_delaware_screening(DESIGN_INPUTS, shell_side_result)
    shell_side_correlation_rating = calculate_shell_side_correlation_rating(DESIGN_INPUTS, shell_side_result, bell_delaware_screening)
    geometry_aware_rating = calculate_geometry_aware_bell_delaware(DESIGN_INPUTS, BELL_DELAWARE_ADVANCED)
    benchmark_df = compare_against_reference(geometry_aware_rating, BELL_DELAWARE_ADVANCED)
    print('Recalculated from transcribed drawing geometry. Review estimates and correlation applicability.')
else:
    print('No recalculation from actual geometry performed; existing results remain illustrative only.')

## 32.4 Trusted reference-rating CSV template

Use one row per matched operating point from a traceable vendor rating, qualified reference implementation, or experimental test. Include source/version/date, geometry and operating-point match flags, nozzle-loss treatment, reviewer details, and agreed tolerances. Do not fabricate reference numbers. The template is created blank if it does not already exist.

In [ ]:
REFERENCE_COLUMNS = [
    'case_id','reference_source','source_version_or_date','reference_report_id',
    'geometry_matched_yes_no','fluid_properties_matched_yes_no','operating_point_matched_yes_no',
    'nozzle_losses_treatment_matched_yes_no','reference_h_shell_W_m2K','reference_dp_shell_kPa',
    'model_h_shell_W_m2K','model_dp_shell_kPa','h_tolerance_percent','dp_tolerance_percent',
    'reviewer_name','reviewer_qualification','review_date_YYYY_MM_DD','notes'
]
reference_csv_path = Path('trusted_reference_rating_comparison.csv')
if not reference_csv_path.exists():
    pd.DataFrame(columns=REFERENCE_COLUMNS).to_csv(reference_csv_path, index=False)
print('Reference comparison template:', reference_csv_path.resolve())

## 32.5 Automated comparison and release gate

This comparison uses the geometry-aware model outputs after any approved geometry reload. It checks traceability, matched-case flags, per-metric error and project-set tolerances. A benchmark `PASS` is only a numerical evidence gate—not a code certification or engineer approval.

In [ ]:
def compare_model_to_trusted_reference(csv_path='trusted_reference_rating_comparison.csv', min_cases=3):
    ref = pd.read_csv(csv_path)
    if ref.empty:
        return ref, {'status':'BLOCKED — NO TRUSTED REFERENCE DATA', 'matched_cases':0, 'complete_cases':0, 'note':'Populate with traceable reference values; never use synthetic values as reference.'}
    missing_cols = sorted(set(REFERENCE_COLUMNS) - set(ref.columns))
    if missing_cols:
        raise ValueError(f'Reference CSV missing columns: {missing_cols}')
    model_h = float(geometry_aware_rating['Corrected_shell_HTC_W_m2K'])
    model_dp = float(geometry_aware_rating['Total_shell_DP_kPa_excluding_nozzles'])
    # If model values are blank in the CSV, use the current model outputs for the current geometry/case.
    ref['model_h_shell_W_m2K'] = pd.to_numeric(ref['model_h_shell_W_m2K'], errors='coerce')
    ref['model_dp_shell_kPa'] = pd.to_numeric(ref['model_dp_shell_kPa'], errors='coerce')
    if len(ref) == 1:
        ref['model_h_shell_W_m2K'] = ref['model_h_shell_W_m2K'].fillna(model_h)
        ref['model_dp_shell_kPa'] = ref['model_dp_shell_kPa'].fillna(model_dp)
    # For multiple operating points, each row must contain the model result for that same point;
    # never compare one current model output against several different reference cases.
    for c in ['reference_h_shell_W_m2K','reference_dp_shell_kPa','h_tolerance_percent','dp_tolerance_percent']:
        ref[c] = pd.to_numeric(ref[c], errors='coerce')
    flags = ['geometry_matched_yes_no','fluid_properties_matched_yes_no','operating_point_matched_yes_no','nozzle_losses_treatment_matched_yes_no']
    matched = ref[flags].astype(str).apply(lambda col: col.str.strip().str.lower()).eq('yes').all(axis=1)
    traceable = ref['reference_source'].fillna('').astype(str).str.strip().ne('') & ref['source_version_or_date'].fillna('').astype(str).str.strip().ne('') & ref['reference_report_id'].fillna('').astype(str).str.strip().ne('')
    reviewer = ref['reviewer_name'].fillna('').astype(str).str.strip().ne('') & ref['reviewer_qualification'].fillna('').astype(str).str.strip().ne('') & ref['review_date_YYYY_MM_DD'].fillna('').astype(str).str.strip().ne('')
    valid = ref['reference_h_shell_W_m2K'].gt(0) & ref['reference_dp_shell_kPa'].gt(0) & ref['model_h_shell_W_m2K'].gt(0) & ref['model_dp_shell_kPa'].gt(0) & ref['h_tolerance_percent'].gt(0) & ref['dp_tolerance_percent'].gt(0)
    ref['h_error_percent'] = (ref['model_h_shell_W_m2K'] - ref['reference_h_shell_W_m2K']).abs() / ref['reference_h_shell_W_m2K'].abs() * 100
    ref['dp_error_percent'] = (ref['model_dp_shell_kPa'] - ref['reference_dp_shell_kPa']).abs() / ref['reference_dp_shell_kPa'].abs() * 100
    ref['evidence_complete'] = matched & traceable & reviewer & valid
    ref['within_tolerance'] = ref['evidence_complete'] & ref['h_error_percent'].le(ref['h_tolerance_percent']) & ref['dp_error_percent'].le(ref['dp_tolerance_percent'])
    complete = int(ref['evidence_complete'].sum())
    matched_count = int(matched.sum())
    passed = complete >= int(min_cases) and bool(ref.loc[ref['evidence_complete'],'within_tolerance'].all())
    summary = {
        'status': 'NUMERICAL BENCHMARK GATE PASS — ENGINEER SIGN-OFF STILL REQUIRED' if passed else 'BLOCKED — INSUFFICIENT / UNMATCHED / OUT-OF-TOLERANCE EVIDENCE',
        'matched_cases': matched_count, 'complete_cases': complete, 'minimum_required_cases': int(min_cases),
        'model_current_case': {'h_shell_W_m2K':model_h, 'dp_shell_kPa':model_dp},
        'physical_validation_certified': False,
        'note': 'Confirm source independence, same geometry/conditions, uncertainty and project tolerances. This is not design certification.'
    }
    return ref, summary

reference_comparison_detail, reference_comparison_summary = compare_model_to_trusted_reference()
reference_comparison_detail.to_csv('trusted_reference_comparison_results.csv', index=False)
Path('trusted_reference_comparison_summary.json').write_text(json.dumps(reference_comparison_summary, indent=2), encoding='utf-8')
print(json.dumps(reference_comparison_summary, indent=2))
if not reference_comparison_detail.empty:
    display(reference_comparison_detail)

## 32.6 Final evidence and release checklist

- [ ] Approved drawing number/revision recorded; geometry transcribed and independently checked.
- [ ] Actual tube layout, tube-row counts, window free area and clearances confirmed from drawing/CAD or qualified rating software.
- [ ] Fluid properties and operating conditions match the reference case.
- [ ] Reference report/version/date and nozzle-loss treatment are traceable.
- [ ] Project-specific tolerances agreed before reviewing errors.
- [ ] Discrepancies investigated; no tuning solely to force a pass.
- [ ] Qualified engineer reviewed thermal, hydraulic, vibration and mechanical design.

**Status remains preliminary until these evidence items and formal sign-off exist.**

# 33. Design-rule and applicability screening
Compares the current design (the built-in example, or the geometry loaded from a verified drawing in §32) against **common screening heuristics** for shell-and-tube exchangers. Ranges are widely used rules of thumb, **not** code requirements: *REVIEW* means "an engineer should look at this", not "non-compliant". TEMA / ASME compliance is not assessed here.

In [ ]:
def design_rule_checks(p, r):
    """Return a table comparing design values with common screening ranges."""
    rows = []

    def _guide(lo, hi, unit):
        u = f" {unit}" if unit else ""
        if lo is not None and hi is not None:
            return f"{lo:g} – {hi:g}{u}"
        return f"≥ {lo:g}{u}" if lo is not None else f"≤ {hi:g}{u}"

    def add(name, value, lo, hi, unit, basis):
        if value is None or not np.isfinite(float(value)):
            status = "NOT EVALUATED"
            shown = float("nan")
        else:
            shown = float(value)
            ok = (lo is None or shown >= lo) and (hi is None or shown <= hi)
            status = "WITHIN" if ok else "REVIEW"
        rows.append({"Check": name, "Value": round(shown, 4) if np.isfinite(shown) else shown,
                     "Screening range": _guide(lo, hi, unit), "Status": status, "Basis": basis})

    add("Tube pitch / tube OD", p["tube_pitch_m"] / p["tube_outer_diameter_m"], 1.25, 1.5, "",
        "TEMA minimum 1.25; common practice up to about 1.5")
    add("Baffle spacing / shell ID", p["baffle_spacing_m"] / p["shell_inside_diameter_m"], 0.2, 1.0, "",
        "Common screening range; TEMA minimum is about one-fifth of shell ID (and not below 50 mm)")
    add("Baffle cut", p["baffle_cut_fraction"], 0.15, 0.45, "",
        "Common segmental baffle range 15–45 % of shell ID")
    add("Tube length / shell ID", p["tube_length_m"] / p["shell_inside_diameter_m"], 3.0, 15.0, "",
        "Wide screening range; about 5–10 is typical")
    add("Tube-side velocity", r.get("Tube_side_velocity_m_s"), 0.9, 2.5, "m/s",
        "Typical for water-like liquids; other fluids differ")
    add("Tube-side Reynolds number", r.get("Reynolds_number_tube_side"), 10000.0, None, "",
        "Dittus–Boelter applicability (fully turbulent flow)")
    add("Thermal area margin", r.get("Area_margin_percent"), p["minimum_thermal_margin_fraction"] * 100.0, 50.0, "%",
        "At least the project minimum; a very large margin suggests conservative U or F assumptions")
    add("LMTD correction factor F", p["lmtd_correction_factor_F"], 0.8, None, "",
        "Common practice F ≥ 0.8; lower values suggest more shells in series")
    add("Minimum terminal temperature difference",
        min(p["hot_inlet_C"] - p["cold_outlet_target_C"], p["hot_outlet_target_C"] - p["cold_inlet_C"]), 5.0, None, "K",
        "Typical minimum approach for shell-and-tube duties")
    return pd.DataFrame(rows)

design_rule_table = design_rule_checks(DESIGN_INPUTS, preliminary_result)
display(design_rule_table)
_n_review = int((design_rule_table["Status"] == "REVIEW").sum())
_n_total = len(design_rule_table)
print(f"\nScreening result: {_n_total - _n_review}/{_n_total} checks within range; {_n_review} flagged for engineering review.")
design_rule_table.to_csv("heat_exchanger_design_rule_screening.csv", index=False)

# 33A. Calculated rating closure (replaces assumed U and F with computed values)

Until now the area check in §22 used an **assumed** clean U (900 W/m²K) and an **assumed** LMTD correction factor F = 0.90. This section closes the loop:

1. **F is computed** from the terminal temperatures and the tube-pass arrangement (Bowman/Mueller/Nagle closed form for one shell pass, even number of tube passes).
2. **U is built from film coefficients**: tube-side *h* (Dittus–Boelter, §22), shell-side *h* (geometry-aware Bell–Delaware screen, §28), both fouling films and tube-wall conduction, all referred to the tube outside area.
3. The **resistance breakdown** shows which film controls, and a **sensitivity sweep** shows how much area margin survives if the unvalidated shell-side h is optimistic.

> Still a screening calculation. The shell-side h inherits every estimate flag from §28 (estimated row counts, placeholder friction factor, estimated window area), so the computed margin is only as good as that coefficient. Nothing here changes the **BLOCKED — PRELIMINARY ONLY** release status.

In [ ]:
def lmtd_correction_factor_1shell(T_hot_in, T_hot_out, T_cold_in, T_cold_out, tube_passes=2, shell_passes=1):
    """LMTD correction factor F for 1 shell pass and an even number of tube passes
    (Bowman-Mueller-Nagle closed form). Returns F in (0, 1]; raises if the duty is
    thermodynamically infeasible for this arrangement."""
    if shell_passes != 1:
        raise ValueError("Closed form implemented for one shell pass only; use a validated method for others.")
    if tube_passes < 2 or tube_passes % 2:
        raise ValueError("Closed form implemented for an even number of tube passes (2, 4, ...).")
    dT_hot = T_hot_in - T_hot_out
    dT_cold = T_cold_out - T_cold_in
    if dT_hot <= 0 or dT_cold <= 0 or T_hot_in <= T_cold_in:
        raise ValueError("Temperatures must give positive hot-side drop and cold-side rise.")
    R = dT_hot / dT_cold
    P = dT_cold / (T_hot_in - T_cold_in)
    if not 0 < P < 1:
        raise ValueError(f"Thermal effectiveness parameter P={P:.3f} must be in (0, 1).")
    if abs(R - 1.0) < 1e-9:
        num = P * np.sqrt(2.0) / (1.0 - P)
        den = np.log((2.0 - P * (2.0 - np.sqrt(2.0))) / (2.0 - P * (2.0 + np.sqrt(2.0))))
    else:
        S = np.sqrt(R * R + 1.0)
        if P * R >= 1.0:
            raise ValueError("P*R >= 1: duty not achievable in a single shell pass.")
        arg_num = (2.0 - P * (R + 1.0 - S))
        arg_den = (2.0 - P * (R + 1.0 + S))
        if arg_den <= 0 or arg_num <= 0:
            raise ValueError("Temperature cross: duty not achievable in a 1-shell-pass exchanger (F undefined).")
        num = S / (R - 1.0) * np.log((1.0 - P) / (1.0 - P * R))
        den = np.log(arg_num / arg_den)
    F = float(num / den)
    if not np.isfinite(F) or F <= 0:
        raise ValueError("F could not be evaluated for these temperatures.")
    return min(F, 1.0)


def overall_U_outside(h_i, h_o, Di, Do, Rf_i=0.0, Rf_o=0.0, k_wall=45.0):
    """Overall U referred to tube OUTSIDE area (W/m2K) and its resistance breakdown (m2K/W)."""
    if min(h_i, h_o, Di, Do, k_wall) <= 0 or Do <= Di or Rf_i < 0 or Rf_o < 0:
        raise ValueError("Film coefficients, diameters and k_wall must be positive; Do > Di; fouling >= 0.")
    parts = {
        "Shell-side film": 1.0 / h_o,
        "Shell-side fouling": Rf_o,
        "Tube wall": Do * np.log(Do / Di) / (2.0 * k_wall),
        "Tube-side fouling": Rf_i * Do / Di,
        "Tube-side film": Do / (Di * h_i),
    }
    return 1.0 / sum(parts.values()), parts


def rating_closure(p, tube_result, shell_result, k_wall=45.0, shell_h_factor=1.0, fouling_factor=1.0):
    """Area required vs installed using computed F and computed U (screening)."""
    h_i = tube_result["tube_side_h"] if "tube_side_h" in tube_result else None
    if h_i is None or not np.isfinite(h_i):
        raise ValueError("Tube-side h unavailable.")
    h_o = float(shell_result["Corrected_shell_HTC_W_m2K"]) * shell_h_factor
    U, parts = overall_U_outside(
        h_i, h_o, p["tube_inner_diameter_m"], p["tube_outer_diameter_m"],
        Rf_i=p["fouling_tube_m2K_W"] * fouling_factor, Rf_o=p["fouling_shell_m2K_W"] * fouling_factor, k_wall=k_wall)
    F = lmtd_correction_factor_1shell(p["hot_inlet_C"], p["hot_outlet_target_C"],
                                      p["cold_inlet_C"], p["cold_outlet_target_C"],
                                      tube_passes=int(p["tube_passes"]), shell_passes=int(p["shell_passes"]))
    lmtd = lmtd_counter_current(p["hot_inlet_C"], p["hot_outlet_target_C"], p["cold_inlet_C"], p["cold_outlet_target_C"])
    Q_kW = min(p["hot_mass_flow_kg_s"] * p["hot_cp_kJ_kgK"] * (p["hot_inlet_C"] - p["hot_outlet_target_C"]),
               p["cold_mass_flow_kg_s"] * p["cold_cp_kJ_kgK"] * (p["cold_outlet_target_C"] - p["cold_inlet_C"]))
    A_req = Q_kW * 1000.0 / (U * F * lmtd)
    A_inst = np.pi * p["tube_outer_diameter_m"] * p["tube_length_m"] * p["tube_count"]
    total = sum(parts.values())
    return {"h_tube_W_m2K": float(h_i), "h_shell_W_m2K": float(h_o), "U_calc_W_m2K": float(U),
            "F_calc": float(F), "LMTD_K": float(lmtd), "Q_kW": float(Q_kW),
            "Area_required_m2": float(A_req), "Area_installed_m2": float(A_inst),
            "Area_margin_percent": float(100.0 * (A_inst - A_req) / A_req),
            "resistance_share_percent": {k: float(100.0 * v / total) for k, v in parts.items()}}


def gnielinski_nusselt(Re, Pr):
    """Gnielinski correlation, valid roughly 3,000 <= Re <= 5e6 and 0.5 <= Pr <= 2000."""
    if not 3000 <= Re <= 5e6 or not 0.5 <= Pr <= 2000:
        raise ValueError("Gnielinski outside its stated range (3,000 <= Re <= 5e6, 0.5 <= Pr <= 2000).")
    f = (0.79 * np.log(Re) - 1.64) ** -2
    return (f / 8.0) * (Re - 1000.0) * Pr / (1.0 + 12.7 * np.sqrt(f / 8.0) * (Pr ** (2.0 / 3.0) - 1.0))

def tube_side_h_for_closure(p, prelim):
    """Tube-side h: Dittus-Boelter for Re >= 10,000, Gnielinski for 3,000 <= Re < 10,000 (flagged)."""
    Re, Pr = prelim["Reynolds_number_tube_side"], prelim["Prandtl_number_tube_side"]
    if Re >= 10000:
        Nu, method = nusselt_dittus_boelter(Re, Pr, heating_fluid=True), "Dittus-Boelter"
    else:
        Nu, method = gnielinski_nusselt(Re, Pr), "Gnielinski (transitional Re: use with caution)"
    return {"tube_side_h": Nu * p["tube_conductivity_W_mK"] / p["tube_inner_diameter_m"], "method": method, "Re": Re, "Nu": Nu}

_tube_for_closure = tube_side_h_for_closure(DESIGN_INPUTS, preliminary_result)
print(f"Tube-side h = {_tube_for_closure['tube_side_h']:.0f} W/m²K via {_tube_for_closure['method']} (Re = {_tube_for_closure['Re']:.0f})")
K_WALL_W_mK = 45.0  # carbon-steel placeholder; set from the actual tube material
try:
    closure = rating_closure(DESIGN_INPUTS, _tube_for_closure, geometry_aware_rating, k_wall=K_WALL_W_mK)
    _assumed = {"U_assumed_fouled_W_m2K": preliminary_result["U_fouled_est_W_m2K"],
                "F_assumed": preliminary_result["LMTD_correction_factor_F"],
                "Area_margin_assumed_percent": preliminary_result["Area_margin_percent"]}
    cmp_df = pd.DataFrame([
        {"Quantity": "Overall U (fouled), W/m²K", "Assumed (§22)": _assumed["U_assumed_fouled_W_m2K"], "Calculated (§33A)": closure["U_calc_W_m2K"]},
        {"Quantity": "LMTD correction factor F", "Assumed (§22)": _assumed["F_assumed"], "Calculated (§33A)": closure["F_calc"]},
        {"Quantity": "Required area, m²", "Assumed (§22)": preliminary_result["Area_required_m2"], "Calculated (§33A)": closure["Area_required_m2"]},
        {"Quantity": "Area margin, %", "Assumed (§22)": _assumed["Area_margin_assumed_percent"], "Calculated (§33A)": closure["Area_margin_percent"]},
    ])
    display(cmp_df.round(3))
    res_df = pd.Series(closure["resistance_share_percent"]).sort_values(ascending=False).rename("Share of total resistance, %").to_frame()
    display(res_df.round(1))
    print(f"Controlling resistance: {res_df.index[0]} ({res_df.iloc[0, 0]:.1f} %)")
    closure_status = "COMPUTED — SCREENING ONLY (shell-side h unvalidated)"
except ValueError as err:
    closure = None
    closure_status = f"NOT EVALUATED: {err}"
    print(closure_status)

## 33A.1 How robust is the area margin?

The shell-side coefficient carries the largest uncertainty (placeholder friction factor, estimated row counts, leakage factor Jl ≈ 0.6). The sweep below scales the shell-side h and the fouling resistances and shows where the **area margin crosses zero**. If the margin disappears within a plausible error band, the design cannot be accepted on this model alone.

In [ ]:
if closure is not None:
    h_factors = np.linspace(0.1, 1.3, 25)
    foul_factors = (0.5, 1.0, 1.5, 2.0)
    rows = []
    for ff in foul_factors:
        for hf in h_factors:
            c = rating_closure(DESIGN_INPUTS, _tube_for_closure, geometry_aware_rating,
                               k_wall=K_WALL_W_mK, shell_h_factor=hf, fouling_factor=ff)
            rows.append({"fouling_factor": ff, "shell_h_factor": hf, "margin_percent": c["Area_margin_percent"]})
    sweep = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for ff, grp in sweep.groupby("fouling_factor"):
        ax.plot(grp["shell_h_factor"], grp["margin_percent"], marker="o", ms=3, label=f"fouling × {ff:g}")
    ax.axhline(0, color="k", lw=1)
    ax.axhline(100 * DESIGN_INPUTS["minimum_thermal_margin_fraction"], color="r", ls="--", lw=1, label="minimum margin target")
    ax.set_xlabel("Shell-side h as fraction of calculated value")
    ax.set_ylabel("Area margin (%)")
    ax.set_title("Area margin vs shell-side h and fouling uncertainty")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig("area_margin_robustness.png", dpi=200, bbox_inches="tight"); plt.show()

    breakeven = {}
    for ff, grp in sweep.groupby("fouling_factor"):
        g = grp.sort_values("shell_h_factor")
        ok = g[g["margin_percent"] >= 100 * DESIGN_INPUTS["minimum_thermal_margin_fraction"]]
        breakeven[f"fouling × {ff:g}"] = (float(ok["shell_h_factor"].min()) if not ok.empty else None)
    be_df = pd.DataFrame({"Min. shell-h fraction that still meets margin target": breakeven})
    display(be_df.round(3))
    print("None = margin target not met anywhere in the swept range.")

## 33A.2 Export the closure result

In [ ]:
closure_report = {
    "status": closure_status,
    "inputs_note": "Tube wall conductivity is a placeholder; shell-side h inherits all §28 estimate flags.",
    "k_wall_W_mK": K_WALL_W_mK,
    "closure": closure,
    "shell_side_estimation_flags": geometry_aware_rating.get("Estimation_flags", []),
    "release_status": "BLOCKED — PRELIMINARY ONLY",
}
Path("rating_closure_report.json").write_text(json.dumps(closure_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Saved rating_closure_report.json")

# 34. Software verification suite
Automated checks that the **code** behaves as intended: closed-form hand calculations, input guards, and the logic of the drawing and reference gates. These are **software tests only**. Passing them is **not** physical validation of the exchanger model, and the test rows used for the reference gate are synthetic and are never evidence.

In [ ]:
import copy, tempfile, math
from pathlib import Path

TEST_RESULTS = []

def _record(name, ok, detail=""):
    TEST_RESULTS.append({"test": name, "result": "PASS" if ok else "FAIL", "detail": detail})

def _raises(fn, exc=ValueError):
    try:
        fn()
    except exc:
        return True
    except Exception:
        return False
    return False

# --- Thermal functions against hand calculations -------------------------
_lmtd = lmtd_counterflow(90, 65, 25, 45)  # dT1 = 45, dT2 = 40
_record("LMTD counterflow hand calc", math.isclose(_lmtd, 5 / math.log(45 / 40), rel_tol=1e-9), f"{_lmtd:.4f} K")
_record("LMTD equal terminal differences", math.isclose(lmtd_counterflow(80, 60, 40, 60), 20.0))
_record("LMTD rejects temperature cross", _raises(lambda: lmtd_counterflow(60, 40, 50, 70)))

_eps = effectiveness_counterflow(1.0, 1.0, 4.184, 4.184, 90, 60, 30, 60)
_record("Effectiveness balanced-flow hand calc", math.isclose(_eps, 0.5), f"{_eps:.4f}")

_perf = exchanger_performance(90, 65, 25, 45, 2.5, 3.125)  # Q_hot = Q_cold = 261.5 kW
_record("Performance: effectiveness within 0-1", 0.0 <= _perf["effectiveness"] <= 1.0, f"{_perf['effectiveness']:.3f}")
_record("Performance: rejects non-positive flow", _raises(lambda: exchanger_performance(90, 65, 25, 45, 0.0, 1.0)))

_record("Reynolds number hand calc", math.isclose(reynolds_number(1000, 1.0, 0.01, 0.001), 1.0e4))
_record("Prandtl number hand calc", math.isclose(prandtl_number(4.184, 0.001, 0.6), 4184 * 0.001 / 0.6))
_nu = nusselt_dittus_boelter(1.0e4, 7.0, heating_fluid=True)
_record("Dittus-Boelter hand calc", math.isclose(_nu, 0.023 * 1.0e4 ** 0.8 * 7.0 ** 0.4), f"Nu = {_nu:.2f}")
_record("Dittus-Boelter rejects laminar/transitional Re", _raises(lambda: nusselt_dittus_boelter(5000, 7.0)))
_record("Fouling-free U hand calc", math.isclose(overall_U_with_fouling(1000, 1000, 0, 0, 0.0, 1.0), 500.0))
_record("Fouling reduces U", overall_U_with_fouling(2500, 1800, 0.0005, 0.0007) < overall_U_with_fouling(2500, 1800, 0, 0))

# --- Simulated-dataset energy balance (section 3) ----------------------
_record("Dataset energy balance (hot side)", float(np.max(np.abs(Q_hot_check - Q))) < 1e-9)
_record("Dataset energy balance (cold side)", float(np.max(np.abs(Q_cold_check - Q))) < 1e-9)

# --- ML sanity: trained model must beat the mean baseline --------------
_r2 = metrics.set_index("Model")["R²"]
_record("Best ML model beats mean baseline", float(_r2.max()) > float(_r2.get("Baseline (mean)", -1)) + 0.5, f"best R² = {float(_r2.max()):.4f}")

# --- Drawing-geometry gate logic (works on a temp file; globals restored) --
_DUMMY_GEOMETRY = {
    "drawing_number": "SOFTWARE-TEST", "drawing_revision": "T", "drawing_date": "2000-01-01",
    "transcribed_by": "test", "independently_checked_by": "test", "units": "SI",
    "geometry_source_status": "DRAWING_TRANSCRIBED_AND_CHECKED",
    "shell_inside_diameter_m": 0.4, "bundle_outer_diameter_m": 0.38,
    "tube_outer_diameter_m": 0.019, "tube_inner_diameter_m": 0.016, "tube_length_m": 3.0,
    "tube_count": 100, "tube_passes": 2, "shell_passes": 1, "tube_pitch_m": 0.0254,
    "tube_layout": "triangular", "baffle_spacing_central_m": 0.15,
    "baffle_cut_fraction": 0.25, "shell_bundle_diametral_clearance_m": 0.02,
    "tube_baffle_hole_diametral_clearance_m": 0.0008,
}
_saved_design = copy.deepcopy(DESIGN_INPUTS)
_saved_bd = copy.deepcopy(BELL_DELAWARE_ADVANCED)
try:
    with tempfile.TemporaryDirectory() as _td:
        def _gate(mod=None):
            g = dict(_DUMMY_GEOMETRY); g.update(mod or {})
            p = Path(_td) / "g.json"; p.write_text(json.dumps(g), encoding="utf-8")
            return load_and_apply_drawing_geometry(str(p))
        _record("Geometry gate: valid data loads", _gate()[1]["status"].startswith("DRAWING INPUTS LOADED"))
        _record("Geometry gate: unchecked status blocks", _gate({"geometry_source_status": "UNVERIFIED_TEMPLATE"})[1]["status"].startswith("BLOCKED"))
        _record("Geometry gate: missing field blocks", _gate({"tube_length_m": None})[1]["status"].startswith("BLOCKED"))
        _record("Geometry gate: placeholder date blocks", _gate({"drawing_date": "YYYY-MM-DD"})[1]["status"].startswith("BLOCKED"))
        _record("Geometry gate: pitch <= tube OD rejected", _raises(lambda: _gate({"tube_pitch_m": 0.019})))
        _record("Geometry gate: bundle OD >= shell ID rejected", _raises(lambda: _gate({"bundle_outer_diameter_m": 0.45})))
        _record("Geometry gate: fractional tube count rejected", _raises(lambda: _gate({"tube_count": 100.5})))
        _record("Geometry gate: baffle cut >= 0.5 rejected", _raises(lambda: _gate({"baffle_cut_fraction": 0.55})))
finally:
    DESIGN_INPUTS.clear(); DESIGN_INPUTS.update(_saved_design)
    BELL_DELAWARE_ADVANCED.clear(); BELL_DELAWARE_ADVANCED.update(_saved_bd)

# --- Reference-comparison gate logic (SYNTHETIC rows: software test only) --
def _ref_rows(n, model_scale=1.0, flag="yes"):
    rows = []
    for i in range(n):
        rows.append({
            "case_id": f"SW-{i}", "reference_source": "software-test", "source_version_or_date": "test-v0",
            "reference_report_id": f"SW-{i}", "geometry_matched_yes_no": flag,
            "fluid_properties_matched_yes_no": "yes", "operating_point_matched_yes_no": "yes",
            "nozzle_losses_treatment_matched_yes_no": "yes",
            "reference_h_shell_W_m2K": 1000.0, "reference_dp_shell_kPa": 10.0,
            "model_h_shell_W_m2K": 1000.0 * model_scale, "model_dp_shell_kPa": 10.0 * model_scale,
            "h_tolerance_percent": 10.0, "dp_tolerance_percent": 15.0,
            "reviewer_name": "test", "reviewer_qualification": "test",
            "review_date_YYYY_MM_DD": "2000-01-01", "notes": "SOFTWARE TEST ONLY - NOT EVIDENCE",
        })
    return pd.DataFrame(rows, columns=REFERENCE_COLUMNS)

with tempfile.TemporaryDirectory() as _td:
    def _cmp(df, min_cases=3):
        p = Path(_td) / "r.csv"; df.to_csv(p, index=False)
        return compare_model_to_trusted_reference(str(p), min_cases=min_cases)[1]["status"]
    _record("Reference gate: 3 in-tolerance cases pass", _cmp(_ref_rows(3)).startswith("NUMERICAL BENCHMARK GATE PASS"))
    _record("Reference gate: out-of-tolerance blocks", _cmp(_ref_rows(3, model_scale=1.3)).startswith("BLOCKED"))
    _record("Reference gate: too few cases blocks", _cmp(_ref_rows(2)).startswith("BLOCKED"))
    _record("Reference gate: unmatched geometry blocks", _cmp(_ref_rows(3, flag="no")).startswith("BLOCKED"))
    _record("Reference gate: empty file blocks", _cmp(pd.DataFrame(columns=REFERENCE_COLUMNS)).startswith("BLOCKED"))


# --- Design-rule screening logic ----------------------------------------
_p_ok = dict(DESIGN_INPUTS); _r_ok = {"Tube_side_velocity_m_s": 1.5, "Reynolds_number_tube_side": 20000.0, "Area_margin_percent": 20.0}
_tbl_ok = design_rule_checks(_p_ok, _r_ok)
_record("Design rules: well-proportioned design has no REVIEW flags", int((_tbl_ok["Status"] == "REVIEW").sum()) == 0)
_p_bad = dict(DESIGN_INPUTS); _p_bad["tube_pitch_m"] = 1.1 * _p_bad["tube_outer_diameter_m"]
_record("Design rules: tight pitch ratio flagged", design_rule_checks(_p_bad, _r_ok).set_index("Check").loc["Tube pitch / tube OD", "Status"] == "REVIEW")
_record("Design rules: low Reynolds number flagged", design_rule_checks(_p_ok, dict(_r_ok, Reynolds_number_tube_side=5000.0)).set_index("Check").loc["Tube-side Reynolds number", "Status"] == "REVIEW")
_record("Design rules: missing value not evaluated", design_rule_checks(_p_ok, {}).set_index("Check").loc["Tube-side velocity", "Status"] == "NOT EVALUATED")

# --- Rating-closure (§33A) logic -----------------------------------------
_F_r1 = lmtd_correction_factor_1shell(100, 60, 20, 60)               # R = 1, P = 0.5
_record("F factor R=1 closed form (P=0.5)", math.isclose(_F_r1, (0.5 * math.sqrt(2) / 0.5) / math.log((2 - 0.5 * (2 - math.sqrt(2))) / (2 - 0.5 * (2 + math.sqrt(2)))), rel_tol=1e-9), f"F = {_F_r1:.4f}")
_record("F factor within (0, 1]", all(0 < lmtd_correction_factor_1shell(100, 100 - d, 20, 20 + d * 0.8) <= 1 for d in (5, 10, 20, 30)))
_record("F factor -> 1 for small P", lmtd_correction_factor_1shell(100, 99, 20, 20.8) > 0.999)
_record("F factor falls as P rises", lmtd_correction_factor_1shell(100, 90, 20, 28) > lmtd_correction_factor_1shell(100, 60, 20, 68))
_record("F factor rejects infeasible duty", _raises(lambda: lmtd_correction_factor_1shell(100, 40, 20, 90)))
_record("F factor rejects odd tube passes", _raises(lambda: lmtd_correction_factor_1shell(100, 60, 20, 50, tube_passes=3)))
_record("Gnielinski hand calc (Re=1e4, Pr=7)", abs(gnielinski_nusselt(1e4, 7.0) - 79.5) < 0.5, f"Nu = {gnielinski_nusselt(1e4, 7.0):.1f}")
_record("Gnielinski rejects laminar Re", _raises(lambda: gnielinski_nusselt(1500, 7.0)))
_U_simple, _parts_simple = overall_U_outside(1000, 1000, 0.0159999, 0.016, 0, 0, k_wall=1e9)
_record("Overall U hand calc (thin wall, no fouling)", math.isclose(_U_simple, 500.0, rel_tol=1e-3), f"{_U_simple:.2f}")
_record("Overall U: fouling lowers U", overall_U_outside(3000, 2000, 0.016, 0.019, 1e-4, 1e-4)[0] < overall_U_outside(3000, 2000, 0.016, 0.019)[0])
_record("Overall U rejects Do <= Di", _raises(lambda: overall_U_outside(1000, 1000, 0.02, 0.019)))
if closure is not None:
    _record("Closure: resistance shares sum to 100 %", math.isclose(sum(closure["resistance_share_percent"].values()), 100.0, rel_tol=1e-9))
    _record("Closure: energy duty matches design basis", math.isclose(closure["Q_kW"], preliminary_result["Q_design_basis_kW"], rel_tol=1e-9))
    _record("Closure: margin consistent with areas", math.isclose(closure["Area_margin_percent"], 100 * (closure["Area_installed_m2"] / closure["Area_required_m2"] - 1), rel_tol=1e-9))
    _worse = rating_closure(DESIGN_INPUTS, _tube_for_closure, geometry_aware_rating, shell_h_factor=0.7)
    _record("Closure: lower shell h reduces margin", _worse["Area_margin_percent"] < closure["Area_margin_percent"])

test_table = pd.DataFrame(TEST_RESULTS)
display(test_table)
_n_fail = int((test_table["result"] == "FAIL").sum())
print(f"\nSoftware verification: {len(test_table) - _n_fail}/{len(test_table)} passed")
assert _n_fail == 0, "One or more software verification tests failed — review the table above."

# 35. Consolidated status dashboard
One-page summary of every gate in this notebook. A green software-check line **does not** imply the model is physically validated: release stays blocked until independent evidence and sign-off exist.

In [ ]:
def _get(name, key, default="not run"):
    obj = globals().get(name)
    return obj.get(key, default) if isinstance(obj, dict) else default

_rel = globals().get("professional_release_gate", {})
status_rows = [
    ("Software verification suite", f"{int((test_table['result']=='PASS').sum())}/{len(test_table)} PASS"),
    ("Drawing geometry gate", _get("drawing_gate", "status")),
    ("Design-rule screening", f"{_n_total - _n_review}/{_n_total} within range; {_n_review} for review"),
    ("Calculated rating closure (§33A)", globals().get("closure_status", "not run")),
    ("Trusted reference benchmark gate", _get("reference_comparison_summary", "status")),
    ("Independent benchmark evidence", _rel.get("independent_benchmark_evidence", "not run")),
    ("Geometry verified against issued drawing", "YES" if _rel.get("geometry_verified_against_issued_drawing") else "NO"),
    ("Qualified engineer sign-off", "YES" if _rel.get("qualified_engineer_signoff") else "NO"),
    ("OVERALL RELEASE STATUS", _rel.get("release_status", "BLOCKED — PRELIMINARY ONLY")),
]
dashboard = pd.DataFrame(status_rows, columns=["Gate", "Status"])
display(dashboard)
dashboard.to_csv("heat_exchanger_status_dashboard.csv", index=False)
print("Saved heat_exchanger_status_dashboard.csv")

# 36. Run manifest (audit trail)
Lists every generated or input data file with its size and SHA-256 hash, together with the environment record and gate statuses, so a specific run can be identified and later verified as unmodified.

In [ ]:
import hashlib

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

_TRACKED_SUFFIXES = {".csv", ".json", ".md", ".png"}
_files = sorted(
    p for p in Path(".").iterdir()
    if p.is_file() and p.suffix.lower() in _TRACKED_SUFFIXES and p.name != "run_manifest.json"
)
manifest_files = [
    {"file": p.name, "size_bytes": p.stat().st_size, "sha256": _sha256(p)} for p in _files
]
run_manifest = {
    "environment": ENVIRONMENT_RECORD,
    "gate_status": dict(zip(dashboard["Gate"], dashboard["Status"])),
    "files": manifest_files,
}
Path("run_manifest.json").write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

display(pd.DataFrame(
    [{"file": f["file"], "size_bytes": f["size_bytes"], "sha256 (first 16)": f["sha256"][:16]} for f in manifest_files]
))
print(f"\nSaved run_manifest.json ({len(manifest_files)} files hashed).")